# PHI/PII De-identification of Scanned Documents — v8 (simple: Azure Surrogate writes every fake)

Replaces PHI/PII **on the image** with the fake values **Azure DeID's `Surrogate` operation** writes, keeping the original font, size, ink colour and layout (handwriting gets a handwriting font). No rule-based fake values, no hybrid.

| Step | Section | What happens |
| --- | --- | --- |
| 1 | 4–6 | **Textract `DetectDocumentText`** reads the page; the page text is rebuilt line by line and every word keeps its word id and pixel box |
| 2 | 11b | **GPT-5.4** reads the image: every filled-in field (label, value, is_phi) + PHI outside fields, matched to the OCR words → `label: value` text, GPT PHI words, GPT not-PHI words, protected labels |
| 3 | 12 | **Azure DeID `Surrogate`** on the page text and on the `label: value` text (2 calls). Every word Azure changed gets Azure's value (page text first). **A word GPT says is not PHI is never changed**, and printed form text / field names are never changed. GPT PHI that Azure changed in neither text is erased |
| 4 | 13 | Each changed word is erased and redrawn in the original font and size; longer or shorter fakes are fitted (free space → move unchanged words along → condense → smaller font) |
| 5 | 13b, batch | side-by-side check, `_deid.png`, `_comparison.png`, `_changes.txt`, `_surrogated.txt`, `_gpt.json`, `_verify.json`, `batch_summary.csv` |

**Two format fixes** (switch off in section 12): an Azure date fake that is not a valid date is replaced by the real date shifted by a fixed number of days (`FIX_INVALID_DATES`), and e-mail addresses get a realistic fake (`EMAIL_FAKES = "realistic"`).

**Cost per page:** one Textract call (+ one for the verification record), one GPT-5.4 call, two Azure DeID calls.

## 1. Import Required Libraries

Unchanged. No new packages are needed: the new cells use only Pillow, NumPy, pandas and openpyxl.

In [ ]:
import boto3        # AWS SDK for Python — Textract OCR
import json          # JSON parsing for API responses
import os            # File path operations
from collections import Counter  # Block type counting
from icp.authenticator.aws.boto3_services import boto3_clients as aws_client
import ssl           # Certificate handling
import sys           # Platform detection
import tempfile      # Temporary file creation
import urllib.error  # HTTP error handling
import urllib.request  # HTTP requests

print("✓ All libraries imported successfully")

## 1b. Debug Printouts

`DEBUG = True` prints what each stage produced for the single image in sections 6, 11b, 12, 13 and 14: OCR words (low-confidence and handwritten ones marked), GPT-5.4 fields and whether each was found among the OCR words, Azure tags, every flagged word with the detectors that flagged it, protected words a detector tried to change (and why it was stopped), original → fake values, and what happened to each word on the image. `DEBUG_BATCH = True` prints the same for every page of the batch.

**The printouts show real PHI.** Use them on your own machine only and clear the notebook outputs before sharing it.

In [ ]:
# Debug printouts for each stage. They contain real PHI: clear the outputs before sharing the notebook.
DEBUG = True            # single-image run (sections 6, 11b, 12, 13, 14)
DEBUG_BATCH = False     # every page of the batch run (very long output)
DEBUG_MAX_ROWS = 120    # rows printed per table

def _dbg_title(t):
    print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)

def _dbg_rows(rows, header):
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) if rows else len(str(h)) for i, h in enumerate(header)]
    widths = [min(w, 60) for w in widths]
    fmt = "  ".join("{:<%d}" % w for w in widths)
    print(fmt.format(*header)); print(fmt.format(*["-" * w for w in widths]))
    for r in rows[:DEBUG_MAX_ROWS]:
        print(fmt.format(*[str(x)[:60] for x in r]))
    if len(rows) > DEBUG_MAX_ROWS:
        print(f"... {len(rows) - DEBUG_MAX_ROWS} more rows")

def debug_ocr(offset_map):
    """Stage 1: what Textract read."""
    _dbg_title("STAGE 1 - OCR (Textract DetectDocumentText): lines as read")
    by_line = {}
    for w in offset_map:
        by_line.setdefault(w['line_idx'], []).append(w)
    rows = []
    for li, ws in sorted(by_line.items()):
        low = [w['text'] for w in ws if w['confidence'] < 70]
        hw = [w['text'] for w in ws if w['handwriting']]
        rows.append((li, f"{min(w['confidence'] for w in ws):.0f}", " ".join(w['text'] for w in ws),
                     ("LOW:" + ",".join(low)) if low else "", ("HAND:" + ",".join(hw)) if hw else ""))
    _dbg_rows(rows, ("line", "min_conf", "text", "low_conf_words", "handwriting"))
    print(f"\nwords: {len(offset_map)} | lines: {len(by_line)} | low-confidence (<70) words: "
          f"{sum(w['confidence'] < 70 for w in offset_map)} | handwritten: {sum(w['handwriting'] for w in offset_map)}")

def debug_vlm(stats, kv_pairs, vlm_hits, protected_ids, word_by_id):
    """Stage 2: what GPT-5.4 returned and how it was matched to OCR words."""
    _dbg_title("STAGE 2 - GPT-5.4: fields returned, and whether each was found among the OCR words")
    _dbg_rows([(r["label"], r["value"], r["category"], r["is_phi"], r["matched"]) for r in stats.get("field_rows", [])],
              ("label", "value", "category", "is_phi", "matched_to_ocr"))
    print("\nfree-text PHI:")
    _dbg_rows([(r["value"], r["category"], r["matched"]) for r in stats.get("free_rows", [])],
              ("value", "category", "matched_to_ocr"))
    print(f"\nprinted form text lines from GPT: {stats.get('printed_lines', 0)} "
          f"(matched: {stats.get('printed_lines_matched', 0)})")
    print(f"protected (never changed) words: {len(protected_ids)} -> "
          + " ".join(sorted({word_by_id[i]['text'] for i in protected_ids if i in word_by_id})[:80]))
    print(f"\nlabel: value lines sent to Azure: {len(kv_pairs)} | GPT PHI hits: {len(vlm_hits)}")
    stats_short = {k: v for k, v in stats.items() if not k.endswith("_rows") and k != "vlm_raw"}
    print("stats:", stats_short)

def debug_tagging(azure_entities, flags, offset_map, blocked):
    """Stage 3: Azure tags, flagged words with their detectors, and blocked attempts on protected words."""
    _dbg_title("STAGE 3a - Azure Surrogate: every word Azure changed (original -> Azure value)")
    _dbg_rows([(a["view"], a["category"], f"{a['confidence']:.2f}", a["text"]) for a in azure_entities],
              ("view", "category", "conf", "text"))
    _dbg_title("STAGE 3b - Words flagged as PHI (all detectors)")
    rows = []
    for w in offset_map:
        fl = flags.get(w['word_id'])
        if fl:
            part = "".join(ch if m else "·" for ch, m in zip(w['text'], fl["mask"]))
            rows.append((w['line_idx'], w['text'], fl["kind"], ",".join(sorted(fl["sources"])), part))
    _dbg_rows(rows, ("line", "word", "kind", "found_by", "phi_part"))
    _dbg_title("STAGE 3c - Protected words a detector tried to change (stopped)")
    by_id = {w['word_id']: w for w in offset_map}
    _dbg_rows([(by_id[b[0]]['text'], b[1], b[2], f"{b[3]:.2f}", b[4]) for b in blocked if b[0] in by_id],
              ("word", "kind", "source", "conf", "why_stopped"))

def debug_surrogates(entities):
    """Stage 4: original -> fake."""
    _dbg_title("STAGE 4 - Surrogates: original -> fake")
    _dbg_rows([(e["kind"], e["original"], e.get("surrogate", ""), ",".join(e["sources"])) for e in entities],
              ("kind", "original", "fake", "found_by"))

def debug_render(actions, word_by_id):
    """Stage 5: what happened on the image."""
    _dbg_title("STAGE 5 - Image: action per PHI word")
    _dbg_rows([(word_by_id[wid]['text'], word_by_id[wid].get('new_text', ''), a) for wid, a in actions.items()
               if wid in word_by_id], ("original", "drawn", "action"))

print("✓ Debug printers ready (DEBUG =", DEBUG, "| DEBUG_BATCH =", DEBUG_BATCH, ")")

## 2. Azure DeID Service Configuration

Unchanged. Optional: add `DEID_SURROGATE_KEY=<any secret string>` to `.env` so the same original always maps to the same fake value across runs and pages.

In [ ]:
# Azure DeID Service Configuration — loaded from .env (keeps secrets out of the notebook)
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in the project root

ENDPOINT = os.environ["AZURE_DEID_ENDPOINT"]
TENANT_ID = os.environ["AZURE_DEID_TENANT_ID"]
SCOPE = os.environ.get("AZURE_DEID_SCOPE", "https://deid.azure.com/.default")
API_VERSION = os.environ.get("AZURE_DEID_API_VERSION", "2024-11-15")

print(f"Endpoint: {ENDPOINT}")
print(f"Tenant ID: {TENANT_ID}")
print(f"API Version: {API_VERSION}")

## 3. Image File Handler and Loader

Unchanged.

In [ ]:
# Image file handler - currently supports PNG, JPG, TIFF images
# PDF and TIFF multi-page support will be added later

SUPPORTED_IMAGE_FORMATS = ['.png', '.jpg', '.jpeg', '.bmp', '.gif']

def load_image(image_path):
    """
    Load and validate an image file.
    
    Args:
        image_path: Full path to the image file
        
    Returns:
        image_path: Validated path to the image
        
    Raises:
        FileNotFoundError: If file doesn't exist
        ValueError: If file format is not supported
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")
    
    file_ext = os.path.splitext(image_path)[1].lower()
    
    if file_ext not in SUPPORTED_IMAGE_FORMATS:
        raise ValueError(f"Unsupported format: {file_ext}. Supported formats: {SUPPORTED_IMAGE_FORMATS}")
    
    print(f"✓ Image loaded: {image_path}")
    return image_path


# Set your image path here
IMAGE_PATH = r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\data\test\CMS 1500 202604230008171_page_1.png"
# Validate the image
IMAGE_PATH = load_image(IMAGE_PATH)

## 4. Call AWS Textract for OCR (basic text detection)

`DetectDocumentText` returns every LINE and WORD with its text, confidence, bounding box, polygon and printed/handwriting type. It does not return form key-value pairs or tables; those are rebuilt from the layout in section 7. `@aws_client.refresh_on_expire` renews AWS credentials. The same `get_ocr` reads the output image for the verification record.

In [ ]:
with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

print(f"Image size: {len(image_bytes)/1024:.1f} KB")
assert len(image_bytes) < 10 * 1024 * 1024, "Image exceeds 10MB sync limit"
# the above step is for checking the image size and it should be less than 10 MB
@aws_client.refresh_on_expire
def get_ocr(image_bytes):
    """Basic OCR: lines + words with boxes, confidence and handwriting flag (Textract DetectDocumentText)."""
    textract = aws_client.get("textract")
    return textract.detect_document_text(Document={'Bytes': image_bytes})

response = get_ocr(image_bytes)
blocks = response['Blocks']
print(f"Total blocks returned: {len(blocks)}")

type_counts = Counter(b['BlockType'] for b in blocks)
print("Block type counts:", dict(type_counts))

## 5. Extract LINE Blocks

Unchanged.

In [ ]:
# Cell 3c — Extract LINE blocks with text, confidence, type, and bounding box location

lines = [b for b in blocks if b['BlockType'] == 'LINE']
print(f"Total lines detected: {len(lines)}\n")

output_lines = []

for line in lines:
    text = line['Text']
    conf = line['Confidence']
    block_type = line['BlockType']
    bbox = line['Geometry']['BoundingBox']  # normalized 0-1 coords: Left, Top, Width, Height

    entry = (
        f"Type: {block_type} | Confidence: {conf:.2f}% | "
        f"Location(Left={bbox['Left']:.4f}, Top={bbox['Top']:.4f}, "
        f"Width={bbox['Width']:.4f}, Height={bbox['Height']:.4f}) | "
        f"Text: {text}"
    )
    print(entry)
    output_lines.append(entry)

# Save to a text file
output_txt_path = "C:\\Users\\RSahu\\Documents\\Azure_Deidentification\\extracted_text_detailed.txt"
with open(output_txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved detailed output to {output_txt_path}")

In [ ]:
lines

## 6. Extract WORD Blocks and Build Offset Map

**Modified.** Same `full_text` and `offset_map` as before; each word now also stores its pixel box, line number, skew angle and whether Textract read it as handwriting.

In [ ]:
# Build the word index: reconstruct the page text line by line and remember, for
# every character range, which WORD box it came from.
# Added vs. the old version: pixel box, polygon, line number, skew angle and handwriting flag.
import math
from PIL import Image

def build_word_index(blocks, img_w, img_h):
    """Returns (full_text, offset_map). offset_map is in reading order."""
    block_map = {b['Id']: b for b in blocks} 
    line_blocks = [b for b in blocks if b['BlockType'] == 'LINE']

    full_text = ""
    offset_map = []

    for line_idx, line in enumerate(line_blocks):
        word_ids = []
        for rel in line.get('Relationships', []):
            if rel['Type'] == 'CHILD':
                word_ids.extend(rel['Ids'])

        for word_id in word_ids:
            word_block = block_map.get(word_id)
            if not word_block or word_block['BlockType'] != 'WORD':
                continue  # skip SELECTION_ELEMENT or other non-word children

            word_text = word_block['Text']
            bbox = word_block['Geometry']['BoundingBox']
            poly = word_block['Geometry'].get('Polygon') or []

            # Skew of this word in degrees (0 = horizontal), from the top edge of its polygon
            angle = 0.0
            if len(poly) >= 2:
                dx = (poly[1]['X'] - poly[0]['X']) * img_w
                dy = (poly[1]['Y'] - poly[0]['Y']) * img_h
                if abs(dx) + abs(dy) > 0:
                    angle = math.degrees(math.atan2(dy, dx))

            start = len(full_text)
            full_text += word_text
            end = len(full_text)

            offset_map.append({
                "idx": len(offset_map),
                "start": start,
                "end": end,
                "text": word_text,
                "bbox": bbox,
                "confidence": word_block['Confidence'],
                "word_id": word_id,
                "line_idx": line_idx,
                "px": (int(math.floor(bbox['Left'] * img_w)),
                       int(math.floor(bbox['Top'] * img_h)),
                       int(math.ceil((bbox['Left'] + bbox['Width']) * img_w)),
                       int(math.ceil((bbox['Top'] + bbox['Height']) * img_h))),
                "poly": [(pt['X'] * img_w, pt['Y'] * img_h) for pt in poly] if len(poly) >= 3 else None,
                "angle": angle,
                "handwriting": word_block.get('TextType') == 'HANDWRITING',
            })
            full_text += " "   # space between words
        full_text += "\n"      # newline between lines

    return full_text, offset_map


with Image.open(IMAGE_PATH) as _im:
    IMG_W, IMG_H = _im.size

full_text, offset_map = build_word_index(blocks, IMG_W, IMG_H)
word_by_id = {e['word_id']: e for e in offset_map}

print(f"Image: {IMG_W}x{IMG_H} px")
print(f"Total words indexed: {len(offset_map)}")
print(f"Handwritten words: {sum(e['handwriting'] for e in offset_map)}")
print(f"Reconstructed text length: {len(full_text)} characters")
if DEBUG:
    debug_ocr(offset_map)

In [ ]:
word_by_id

## 8. Label: Value View for Azure (function only)

Section 7 (manual layout pairing) has been removed in this version: label → value pairs come only from GPT-5.4 (section 11b).

`build_context_view()` writes one `label: value` line per pair and records where each value word sits, so Azure's hits can be mapped back to word boxes. It is called in section 11b with the pairs from GPT-5.4.

In [ ]:
# Second view of the page for Azure: one "label: value" line per form field and per
# table cell. Dense forms (CMS-1500, UB-04) print the value far from its label, so the
# line-order text alone gives the model little context. Here each value keeps its
# offsets, so whatever Azure tags can be mapped back to the exact word boxes.

def build_context_view(kv_pairs, table_cells, word_by_id):
    ctx_text, ctx_map = "", []

    def add(label, word_ids):
        nonlocal ctx_text
        ids = [i for i in word_ids if i in word_by_id]
        if not ids:
            return
        ctx_text += (label.strip() or "Value") + ": "
        for wid in ids:
            start = len(ctx_text)
            ctx_text += word_by_id[wid]['text']
            ctx_map.append({"start": start, "end": len(ctx_text), "word_id": wid})
            ctx_text += " "
        ctx_text += "\n"

    for p in kv_pairs:
        add(p['key_text'], p['value_word_ids'])
    for c in table_cells:
        add(c['header_text'], c['word_ids'])
    return ctx_text, ctx_map

## 9. Configure SSL/TLS CA Bundle

Unchanged.

In [ ]:
import certifi

def export_netskope_ca():
    """Write a CA bundle containing certifi roots plus any Netskope roots from the Windows store."""
    base_bundle = certifi.where()
    bundle_text = ""
    if base_bundle and os.path.exists(base_bundle):
        with open(base_bundle, "r", encoding="ascii", errors="ignore") as f:
            bundle_text = f.read()
    if sys.platform != "win32":
        return base_bundle if bundle_text else None

    pems = []
    for store in ("ROOT", "CA"):
        for der, encoding, _ in ssl.enum_certificates(store):
            if encoding != "x509_asn":
                continue
            pem = ssl.DER_cert_to_PEM_cert(der)
            if b"netskope" in der.lower() or b"goskope" in der.lower():
                pems.append(pem)

    if not pems:
        return base_bundle if bundle_text else None

    path = os.path.join(tempfile.gettempdir(), "NetskopeRootPlusCertifi.pem")
    with open(path, "w", encoding="ascii") as f:
        if bundle_text:
            f.write(bundle_text.rstrip() + "\n")
        f.write("\n".join(pems))
    return path

def configure_ca_bundle(explicit=None):
    bundle = explicit or os.environ.get("REQUESTS_CA_BUNDLE") or os.environ.get("SSL_CERT_FILE") or os.environ.get("AWS_CA_BUNDLE")
    if not bundle:
        bundle = export_netskope_ca()
        if bundle:
            print(f"Using CA bundle at {bundle}")
    if bundle:
        os.environ["REQUESTS_CA_BUNDLE"] = bundle
        os.environ["SSL_CERT_FILE"] = bundle
        os.environ["AWS_CA_BUNDLE"] = bundle
    else:
        print("No custom CA bundle in use (none provided, none found).")
    return bundle

# Force a freshly generated bundle so the notebook does not keep reusing an older temp path.
ca_bundle = configure_ca_bundle(export_netskope_ca())
print(f"✓ CA bundle configured: {ca_bundle if ca_bundle else 'Default'}")

## 10. Azure Authentication

Unchanged.

In [ ]:
from azure.identity import InteractiveBrowserCredential

print("Opening browser for sign-in...")
print("(A browser window will open. Login with your Azure account that has DeID Data Owner role)\n")

credential = InteractiveBrowserCredential(tenant_id=TENANT_ID, timeout=240)

try:
    token = credential.get_token(SCOPE)
    print("✓ Authentication: OK")
    print(f"Token expires in: {token.expires_on}")
except Exception as exc:
    print(f"❌ FAILED: authentication error:\n{exc}")
    raise

## 11. Azure DeID Call

`call_azure_deidentification(text, "Surrogate")` sends one request and returns Azure's JSON (`outputText` = the text with every PHI value replaced). Every call is counted in `AZURE_USAGE` (calls, bytes) for the cost. `_split_chunks` cuts text over 20,000 characters at line breaks.

In [ ]:
# Azure DeID call. Fixes vs. the old version:
#   * the request field is "operation" (the old "operationType" was ignored, so the
#     service always ran its default, Surrogate)
#   * a fresh token is requested on every call (azure-identity caches and renews it),
#     which removes the 401 after the first token expires
#   * the full JSON response is returned, so the Tag entities are available

AZURE_MAX_CHARS = 20000   # text is sent in chunks of at most this many characters
AZURE_USAGE = {"calls": 0, "input_bytes": 0}   # billed data: counted before every DeID call

def call_azure_deidentification(text, operation="Surrogate"):
    """Call the Azure DeID service. operation: "Tag", "Redact" or "Surrogate". Returns the JSON dict."""
    AZURE_USAGE["calls"] += 1
    AZURE_USAGE["input_bytes"] += len(text.encode("utf-8"))   # size of the text actually sent
    token = credential.get_token(SCOPE)
    body = json.dumps({"inputText": text, "operation": operation}).encode()
    request = urllib.request.Request(
        f"{ENDPOINT}/deid?api-version={API_VERSION}",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token.token}",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read())
    except urllib.error.HTTPError as exc:
        print(f"❌ Azure DeID error: {exc.code} {exc.reason}")
        print(exc.read().decode(errors="replace")[:500])
        raise


def _split_chunks(text, limit):
    """Split on line boundaries so no entity is cut in half."""
    chunks, pos = [], 0
    while pos < len(text):
        end = min(len(text), pos + limit)
        if end < len(text):
            nl = text.rfind("\n", pos, end)
            if nl > pos:
                end = nl + 1
        chunks.append((pos, text[pos:end]))
        pos = end
    return chunks

print("✅ Azure DeID function defined (Surrogate, used in section 12)")

## 11b. VLM Context (GPT-5.4)

**New in the VLM version.** Basic OCR gives positions but not context. GPT-5.4 looks at the page image and lists every filled-in field: the printed **label**, the **value** copied exactly as printed, a **category**, and whether it is **PHI**. Those values are matched to the Textract words, so every VLM answer lands on exact pixel boxes.

| Step | What happens |
| --- | --- |
| 1 | `vlm_extract()` sends the image to the GPT-5.4 deployment (Azure OpenAI; `endpoint`, `deployment`, `subscription_key`, `api_version` from `.env`) |
| 2 | `match_text()` finds each label and value among the OCR words (same line, n−1…n+1 words, ≥ 85 % similar after removing case and punctuation; values of ≤ 3 characters must match exactly). When a label is found, the value occurrence closest to it is used |
| 3 | The VLM pairs are `kv_pairs`; VLM labels become the protected label words |
| 4 | `ctx_text` / `ctx_map` are rebuilt from these pairs for Azure DeID |
| 5 | Values the VLM marked as PHI → `vlm_hits`, used by section 12 as one more detector (source `vlm`) |

The VLM never decides positions and never replaces the other detectors: Azure Tag on both texts, the label rules, the patterns and propagation all still run. If the call fails (network, timeout, invalid JSON) the page has no label context: Azure Tag on the full page text, the patterns and propagation still run, and the batch summary records the error.

Settings: `USE_VLM`, `VLM_MAX_WIDTH` (image width sent), `VLM_MAX_TOKENS`, `VLM_MATCH_THRESHOLD`, `VLM_CACHE_DIR` (optional: saves each page's VLM answer so a re-run does not call GPT again; the files contain PHI, keep the folder protected).

In [ ]:
# GPT-5.4 (Azure OpenAI) reads the page image and returns label / value / category for every filled-in field.
# Values are matched back to the Textract words, so the VLM adds CONTEXT, while Textract keeps the POSITIONS.
import base64
import io
import re
from difflib import SequenceMatcher
from pathlib import Path

from openai import AzureOpenAI
from PIL import Image

# ---------------------------------------------------------------- settings
USE_VLM = True
VLM_MAX_WIDTH = 2000          # image is resized to at most this width before sending
VLM_MAX_TOKENS = 16000        # GPT-5.x counts reasoning tokens here too
VLM_MATCH_THRESHOLD = 0.85    # how similar a VLM value must be to OCR words to count as the same text
VLM_CACHE_DIR = None          # e.g. Path(r"C:\...\vlm_cache") to reuse answers on re-runs (contains PHI)

# ---------------------------------------------------------------- Azure OpenAI settings from .env
AZURE_OPENAI_ENDPOINT = os.getenv("endpoint") or os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("deployment") or os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_KEY = os.getenv("subscription_key") or os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("api_version") or os.getenv("AZURE_OPENAI_API_VERSION") or "2024-10-21"
VLM_MODEL_NAME = os.getenv("model_name") or os.getenv("model") or "gpt-5.4"     # for logging only

_missing = [n for n, v in {"endpoint": AZURE_OPENAI_ENDPOINT, "deployment": AZURE_OPENAI_DEPLOYMENT,
                           "subscription_key": AZURE_OPENAI_API_KEY}.items() if not v]
if USE_VLM and _missing:
    raise ValueError(f"Missing Azure OpenAI settings in .env: {', '.join(_missing)} (or set USE_VLM = False)")

vlm_client = AzureOpenAI(api_version=AZURE_OPENAI_API_VERSION, azure_endpoint=AZURE_OPENAI_ENDPOINT,
                         api_key=AZURE_OPENAI_API_KEY) if USE_VLM else None

VLM_PROMPT = """You are reading a scanned healthcare document (for example a CMS-1500 or UB-04 claim form).
Return JSON only, in this shape:
{"fields": [{"label": "...", "value": "...", "category": "...", "is_phi": true}],
 "free_text_phi": [{"value": "...", "category": "..."}],
 "printed_text": ["..."]}

fields: every FILLED-IN field on the page.
- "label": the printed field name exactly as printed, including its box or form-locator number
  (e.g. "2. PATIENT'S NAME (Last Name, First Name, Middle Initial)", "10 BIRTHDATE", "45 SERV. DATE").
- "value": the typed or handwritten value for that field, copied EXACTLY as printed: same spelling, spacing,
  punctuation and date format. Do not correct, reformat or complete anything.
- One entry per field. For tables (e.g. UB-04 service lines) give one entry per filled cell that holds an
  identifier or a date, with the column header as label.
- If one box holds a full address, give street, city, state and zip as separate entries with the same label.
- "category": one of name, date, age_over_89, phone, fax, email, ssn, medical_record_number, health_plan_id,
  account_number, license_number, id_number, street, city, state, zip, country, organization, hospital,
  profession, url, other.
- "is_phi": true for identifiers of a patient, insured person, guarantor, relative or provider (names, dates,
  addresses, phone numbers, ID / account / policy / member / license / NPI numbers); false for diagnosis or
  procedure codes, charges, units, checkbox marks, yes/no, sex, relationship, and printed form text.

free_text_phi: identifiers that are NOT in a labelled field (notes, stamps, signatures, headers), copied exactly.

printed_text: every PRE-PRINTED piece of text that belongs to the blank form itself, one entry per printed line,
copied exactly: form titles, approval lines, box captions and labels, checkbox captions (e.g. MEDICARE, MEDICAID,
TRICARE, SSN, EIN, YES, NO), column headers, date captions (MM DD YY), instructions, footers, form numbers.
Never include anything that was typed or written into the form.

Do not list empty fields. Do not invent values.
Return {"fields": [], "free_text_phi": [], "printed_text": []} if nothing is filled in."""

# VLM category -> kind used by section 12 / 13
VLM_KIND = {
    "name": "name", "date": "date", "phone": "phone", "fax": "phone", "email": "email",
    "street": "street", "city": "city", "state": "state", "zip": "zip", "country": "country",
    "organization": "org", "hospital": "org", "profession": "profession",
}   # everything else (ssn, *_number, health_plan_id, age_over_89, url, other) -> "id"


def vlm_kind(category, value=""):
    c = str(category or "").strip().lower()
    if c == "age_over_89":
        digits = re.sub(r"\D", "", str(value))
        return "id" if digits and int(digits) > 89 else None
    return VLM_KIND.get(c, "id")


# ---------------------------------------------------------------- calling GPT-5.4
def _image_b64(image_path, max_width=VLM_MAX_WIDTH):
    image = Image.open(image_path).convert("RGB")
    if image.width > max_width:
        scale = max_width / image.width
        image = image.resize((int(image.width * scale), int(image.height * scale)), Image.Resampling.LANCZOS)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def _parse_json(text_output):
    cleaned = (text_output or "").strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, re.S)
        if not match:
            raise
        return json.loads(match.group(0))


def vlm_extract(image_path):
    """GPT-5.4 answer for one page: {"fields": [...], "free_text_phi": [...]}. Uses the cache if set."""
    cache = Path(VLM_CACHE_DIR) / f"{Path(image_path).stem}_vlm.json" if VLM_CACHE_DIR else None
    if cache and cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))
    response = vlm_client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You read healthcare document images and return strict JSON only."},
            {"role": "user", "content": [
                {"type": "text", "text": VLM_PROMPT},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{_image_b64(image_path)}",
                                                    "detail": "high"}},
            ]},
        ],
        max_completion_tokens=VLM_MAX_TOKENS,
    )
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise RuntimeError(f"VLM output cut off at {VLM_MAX_TOKENS} tokens; raise VLM_MAX_TOKENS")
    result = _parse_json(choice.message.content)
    result.setdefault("fields", []); result.setdefault("free_text_phi", []); result.setdefault("printed_text", [])
    if cache:
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


# ---------------------------------------------------------------- matching VLM text to OCR words
def _compact(s):
    return re.sub(r"[^a-z0-9]", "", str(s or "").lower())

def _similar(a, b):
    if not a or not b:
        return 0.0
    if min(len(a), len(b)) <= 3:
        return 1.0 if a == b else 0.0
    return SequenceMatcher(None, a, b).ratio()

# Date captions printed between date boxes ("MM 05 DD 08 YY 1985") are skipped when matching values.
CAPTION_TOKENS = {"MM", "DD", "YY", "YYYY", "CCYY"}

def _ocr_lines(offset_map):
    by_line = {}
    for w in offset_map:
        if w['text'].upper().strip(".,:;()|[]") in CAPTION_TOKENS:
            continue
        by_line.setdefault(w['line_idx'], []).append(w)
    return [sorted(ws, key=lambda w: w['idx']) for _, ws in sorted(by_line.items())]

def match_text(text_value, offset_map, lines=None):
    """Where `text_value` is printed: [(score, [words])], best first, non-overlapping.
    Windows of n-1..n+1 consecutive words on one OCR line are compared after removing case and punctuation."""
    target = _compact(text_value)
    if not target:
        return []
    n = max(1, len(str(text_value).split()))
    hits = []
    for line in (lines or _ocr_lines(offset_map)):
        for i in range(len(line)):
            for size in range(max(1, n - 1), n + 2):
                win = line[i:i + size]
                if len(win) < size:
                    break
                wc = _compact(" ".join(w['text'] for w in win))
                score = _similar(target, wc)
                if score < VLM_MATCH_THRESHOLD and size == 1 and len(target) >= 4 and target in wc:
                    score = VLM_MATCH_THRESHOLD          # value glued to other text, e.g. "DOB:03/15/1980"
                if score >= VLM_MATCH_THRESHOLD:
                    hits.append((score, -len(win), win))
    chosen, used = [], set()
    for score, _neg, win in sorted(hits, key=lambda h: (-h[0], h[1])):
        ids = {w['word_id'] for w in win}
        if ids & used:
            continue
        used |= ids
        chosen.append((score, win))
    return chosen

def _centre(win):
    return (sum((w['px'][0] + w['px'][2]) / 2 for w in win) / len(win),
            sum((w['px'][1] + w['px'][3]) / 2 for w in win) / len(win))

def _char_spans(win, value):
    """Per word: the characters that hold the value (whole word, or the part inside e.g. 'DOB:03/15/1980')."""
    if len(win) == 1:
        i = win[0]['text'].lower().find(str(value).strip().lower())
        if i >= 0:
            return [(i, i + len(str(value).strip()))]
    return [None] * len(win)


# Printed text of the blank CMS-1500 / UB-04 that must never change, even if GPT does not list it.
BOILERPLATE_PHRASES = [
    "HEALTH INSURANCE CLAIM FORM", "APPROVED BY NATIONAL UNIFORM CLAIM COMMITTEE", "NUCC", "PICA",
    "MEDICARE", "MEDICAID", "TRICARE", "CHAMPVA", "GROUP HEALTH PLAN", "FECA BLK LUNG", "OTHER",
    "(Medicare#)", "(Medicaid#)", "(ID#/DoD#)", "(Member ID#)", "(ID#)",
    "RESERVED FOR NUCC USE", "SSN EIN", "SSN", "EIN", "NPI", "PLEASE PRINT OR TYPE",
    "NUCC Instruction Manual available at", "www.nucc.org", "APPROVED OMB-0938-1197 FORM 1500",
    "SIGNATURE OF PHYSICIAN OR SUPPLIER", "INCLUDING DEGREES OR CREDENTIALS", "SIGNED", "DATE",
    "MM DD YY", "MM DD YYYY", "MM", "DD", "YY", "YYYY", "CCYY", "PATIENT AND INSURED INFORMATION",
    "PHYSICIAN OR SUPPLIER INFORMATION", "CARRIER", "UB-04", "NUBC", "National Uniform Billing Committee",
    "OMB APPROVAL PENDING", "THE CERTIFICATIONS ON THE REVERSE APPLY TO THIS BILL AND ARE MADE A PART HEREOF",
]

def boilerplate_word_ids(offset_map, lines=None):
    """OCR words that are part of known printed form text (every occurrence)."""
    lines = lines or _ocr_lines(offset_map)
    ids = set()
    for phrase in BOILERPLATE_PHRASES:
        for _score, win in match_text(phrase, offset_map, lines):
            ids |= {w['word_id'] for w in win}
    return ids

def _printed_word_ids(printed, offset_map, lines):
    """OCR words of the printed lines GPT listed; long lines are also matched in 3-word pieces."""
    ids, matched = set(), 0
    for p in printed:
        p = str(p or "").strip()
        if not _compact(p):
            continue
        found = match_text(p, offset_map, lines)
        words = p.split()
        if not found and len(words) > 3:
            for k in range(0, len(words) - 2):
                found += match_text(" ".join(words[k:k + 3]), offset_map, lines)
        matched += bool(found)
        for _score, win in found:
            ids |= {w['word_id'] for w in win}
    return ids, matched


def vlm_pairs_and_hits(vlm, offset_map):
    """VLM answer -> (pairs in kv_pairs shape, PHI hits for section 12, protected word ids, stats)."""
    lines = _ocr_lines(offset_map)
    pairs, hits, label_ids = [], [], set()
    stats = {"vlm_fields": 0, "vlm_fields_matched": 0, "vlm_free_text": 0, "vlm_free_text_matched": 0,
             "field_rows": [], "free_rows": []}

    for f in vlm.get("fields", []):
        label, value = str(f.get("label", "") or "").strip(), str(f.get("value", "") or "").strip()
        lab_hits = match_text(label, offset_map, lines) if _compact(label) else []
        lab_win = lab_hits[0][1] if lab_hits else []
        lab_ids = {w['word_id'] for w in lab_win}
        label_ids |= lab_ids                     # every label the VLM names is printed form text: protect it
        if not _compact(value):
            continue
        stats["vlm_fields"] += 1
        row = {"label": label, "value": value, "category": f.get("category"), "is_phi": f.get("is_phi", True),
               "matched": "NO"}
        stats["field_rows"].append(row)
        val_hits = [(s, w) for s, w in match_text(value, offset_map, lines) if not ({x['word_id'] for x in w} & lab_ids)]
        if not val_hits:
            continue                                                     # not found among OCR words
        if lab_win:                                                      # the occurrence closest to its label
            lx, ly = _centre(lab_win)
            win = min(val_hits, key=lambda h: (_centre(h[1])[0] - lx) ** 2 + (_centre(h[1])[1] - ly) ** 2)[1]
        else:
            win = val_hits[0][1]
        stats["vlm_fields_matched"] += 1
        row["matched"] = "yes: " + " ".join(w['text'] for w in win)
        pairs.append({"key_text": label or "Value", "key_word_ids": [w['word_id'] for w in lab_win],
                      "value_text": " ".join(w['text'] for w in win), "value_word_ids": [w['word_id'] for w in win]})
        kind = vlm_kind(f.get("category"), value)
        if f.get("is_phi", True) and kind:
            hits.append({"word_ids": [w['word_id'] for w in win], "spans": _char_spans(win, value), "kind": kind})

    for t in vlm.get("free_text_phi", []):
        value = str(t.get("value", "") or "").strip()
        kind = vlm_kind(t.get("category"), value)
        if not _compact(value) or not kind:
            continue
        stats["vlm_free_text"] += 1
        found = match_text(value, offset_map, lines)
        if len(_compact(value)) < 4:
            found = found[:1]                                            # short values: best match only
        for _score, win in found:
            hits.append({"word_ids": [w['word_id'] for w in win], "spans": _char_spans(win, value), "kind": kind})
        stats["vlm_free_text_matched"] += bool(found)
        stats["free_rows"].append({"value": value, "category": t.get("category"), "matched": "yes" if found else "NO"})

    printed_ids, matched = _printed_word_ids(vlm.get("printed_text", []), offset_map, lines)
    label_ids |= printed_ids
    stats["printed_lines"] = len(vlm.get("printed_text", []))
    stats["printed_lines_matched"] = matched
    return pairs, hits, label_ids, stats


def vlm_build_context(image_path, offset_map, word_by_id):
    """Context for section 12 from GPT-5.4 only (no layout pairs mixed in):
    (kv_pairs, label_word_ids, ctx_text, ctx_map, vlm_hits, vlm_label_ids, stats). Raises if the GPT call fails."""
    vlm = vlm_extract(image_path)
    pairs, hits, vlm_label_ids, stats = vlm_pairs_and_hits(vlm, offset_map)
    stats["vlm_raw"] = vlm                                  # GPT's answer as returned (saved in <name>_gpt.json)
    value_ids = {i for p in pairs for i in p['value_word_ids']} | {i for h in hits for i in h['word_ids']}
    label_ids = ({i for p in pairs for i in p['key_word_ids']} | vlm_label_ids) - value_ids
    ctx_t, ctx_m = build_context_view(pairs, [], word_by_id)
    return pairs, label_ids, ctx_t, ctx_m, hits, vlm_label_ids - value_ids, stats


# Field-name words that are never changed, in any letter case ("Claim Number", "CLAIM NBR", "vendor nbr:", "Acct#").
# A word is protected when, after upper-casing and removing punctuation at its ends, it is exactly one of these.
# Add your own field names here. (Values that merely CONTAIN such a word, e.g. "Nbr:12345", are not protected.)
LABEL_KEYWORDS = {
    "CLAIM", "CLAIMS", "CLM", "NUMBER", "NUMBERS", "NBR", "NUM", "NO", "NOS", "#", "VENDOR", "VNDR", "VEND",
    "SUPPLIER", "MEMBER", "MBR", "SUBSCRIBER", "SUBSCR", "POLICY", "POL", "ACCOUNT", "ACCT", "ACC", "INVOICE", "INV",
    "REF", "REFERENCE", "AUTHORIZATION", "AUTH", "PRIOR", "CONTROL", "CTRL", "GROUP", "GRP", "PLAN", "PAYER", "PAYOR",
    "PATIENT", "PT", "PROVIDER", "PROV", "PHYSICIAN", "INSURED", "INSURANCE", "ID", "I.D", "IDENTIFIER", "NPI", "TIN",
    "TAX", "EIN", "SSN", "MRN", "RECORD", "MEDICAL", "CHART", "DATE", "DATES", "DOB", "DOS", "BIRTH", "BIRTHDATE",
    "ADMIT", "ADMISSION", "DISCHARGE", "SERVICE", "SERVICES", "FROM", "THROUGH", "THRU", "TO", "NAME", "NAMES",
    "FIRST", "LAST", "MIDDLE", "ADDRESS", "ADDR", "STREET", "CITY", "STATE", "ZIP", "ZIPCODE", "CODE",
    "PHONE", "TEL", "TELEPHONE", "FAX", "MOBILE", "CELL", "EMAIL", "E-MAIL", "MAIL", "SIGNATURE", "SIGNED",
    "AMOUNT", "AMT", "TOTAL", "BILLED", "PAID", "CHARGES", "STATUS", "TYPE", "SEX", "GENDER", "AGE",
}

def label_keyword_ids(offset_map):
    ids = set()
    for w in offset_map:
        core = w['text'].upper().strip(".,:;()|[]*-#")
        if core in LABEL_KEYWORDS or (core == "" and w['text'].strip() == "#"):
            ids.add(w['word_id'])
    return ids


def protected_form_ids(offset_map, vlm_label_ids, value_ids=()):
    """Words that are never changed: GPT labels + GPT printed text (minus GPT values) + known boilerplate
    + field-name keywords (LABEL_KEYWORDS) in any letter case, except words GPT returned as a VALUE
    (e.g. "ID" in the STATE box is Idaho, not a label)."""
    return set(vlm_label_ids) | boilerplate_word_ids(offset_map) | (label_keyword_ids(offset_map) - set(value_ids))


def gpt_value_ids(kv_pairs, vlm_hits):
    return {i for p in kv_pairs for i in p['value_word_ids']} | {i for h in vlm_hits for i in h['word_ids']}


# ---------------------------------------------------------------- run for this page
tables, table_cells, header_word_ids = [], [], set()          # basic OCR: no tables
kv_pairs, label_word_ids, ctx_text, ctx_map = [], set(), "", []
vlm_hits, vlm_label_ids, vlm_stats = [], set(), {}
if USE_VLM:
    try:
        kv_pairs, label_word_ids, ctx_text, ctx_map, vlm_hits, vlm_label_ids, vlm_stats = vlm_build_context(
            IMAGE_PATH, offset_map, word_by_id)
        form_fields = {p['key_text']: p['value_text'] for p in kv_pairs if p['key_text'] and p['value_text']}
        print(f"VLM ({VLM_MODEL_NAME}, deployment {AZURE_OPENAI_DEPLOYMENT}): "
              f"{ {k: v for k, v in vlm_stats.items() if not k.endswith('_rows') and k != 'vlm_raw'} }")
        print(f"Pairs for Azure context: {len(kv_pairs)} | protected label words: {len(label_word_ids)} | "
              f"VLM PHI hits: {len(vlm_hits)}")
    except Exception as exc:
        print(f"⚠ VLM step failed, continuing without label context: {exc!r}")
else:
    print("USE_VLM = False: no label context; Azure, patterns and propagation still run.")

vlm_label_ids = protected_form_ids(offset_map, vlm_label_ids, gpt_value_ids(kv_pairs, vlm_hits))      # + known CMS-1500 / UB-04 boilerplate
print(f"Protected form words (never changed): {len(vlm_label_ids)}")
if DEBUG:
    debug_vlm(vlm_stats, kv_pairs, vlm_hits, vlm_label_ids, word_by_id)

## 12. What Changes: Azure Surrogate + GPT's PHI Verdict

| Step | What happens |
| --- | --- |
| 1 | **Azure Surrogate** on the page text and on GPT's `label: value` text. Each answer is lined up with what was sent, line by line, word by word (`_align_view`); every word Azure changed gets Azure's value with the original's punctuation and letter case. The page text wins; the `label: value` text fills in words the page text left unchanged. The same value gets the same fake across the batch (`SURROGATE_MEMO`) |
| 2 | **Never changed:** a word GPT returned as a value with `is_phi: false` (codes, charges, sex, checkboxes…), even if Azure changed it; and protected printed text (GPT labels, CMS-1500 / UB-04 boilerplate, `FORM_TOKENS`, field names in `LABEL_KEYWORDS` in any case) |
| 3 | **GPT PHI that Azure changed in neither text:** erased (`MISSED_PHI_ACTION = "erase"`, only the PHI characters) or kept (`"keep"`) |
| 4 | **Format fixes:** if Azure's fake for a date is not a valid date with the same digits (`05311968` → `48213907`), the real date is shifted by `DATE_SHIFT_DAYS` in the same format (`05311968` → `01261968`, `FIX_INVALID_DATES`); e-mail addresses get a realistic fake from real names and a common provider (`EMAIL_FAKES = "realistic"`; `"azure"` keeps Azure's) |

Debug: Stage 3t (both texts sent and Azure's outputs), Stage 3 (original → fake, and where it came from), 3b (kept because GPT says not PHI), 3c (kept as form text), 3d (GPT PHI Azure missed).

**Saved per page** (next to `_deid.png`): `_ocr_text_sent.txt` / `_ocr_text_deid.txt` (the OCR text sent to Azure and Azure's answer), `_gpt_text_sent.txt` / `_gpt_text_deid.txt` (the GPT `label: value` text sent and Azure's answer), `_changes.txt`, `_surrogated.txt`, `_gpt.json`, and **`_deid_report.xlsx`** with four sheets: **Summary**, **OCR text** (line by line: sent / Azure output / changed), **GPT text** (each GPT field: label, value, category, is_phi, found in OCR, sent / Azure output / changed), **Word decisions** (each word: original, Azure's value from each text, GPT's verdict, what was drawn, and why). A batch also writes `batch_changes.xlsx` (all pages' word decisions). These files hold real PHI: keep them protected.

**Settings:** `SAVE_EXCEL_REPORT`, `MISSED_PHI_ACTION`, `FIX_INVALID_DATES`, `EMAIL_FAKES`, `REUSE_SURROGATES`, `MATCH_CASE`, `SAVE_TEXT_FILES`; `DEID_SURROGATE_KEY` in `.env` keeps the date shift the same across runs.

In [ ]:
# What changes on the image (v8, simple):
#   * Azure DeID SURROGATE on the page text and on GPT's "label: value" text writes every fake value
#     (page text first; the label: value text only for words the page text left unchanged)
#   * a word GPT returned as NOT PHI is never changed, even if Azure changed it
#   * printed form text / field names (GPT labels, boilerplate, LABEL_KEYWORDS) are never changed
#   * GPT PHI that Azure changed in neither text is erased (MISSED_PHI_ACTION)
#   * format fixes: an Azure date fake that is not a valid date -> the original date shifted; e-mails realistic
import re
import hmac
import hashlib
import random
from datetime import datetime, timedelta
# ---------------------------------------------------------------- Azure Surrogate on a text, mapped back to the OCR words
from difflib import SequenceMatcher

MATCH_CASE = True              # Azure's fake in the original's letter case ("SMITH" -> "JOHNSON")
REUSE_SURROGATES = True        # same original value -> same Azure fake on every page of the batch
SURROGATE_MEMO = {}            # compact original value -> Azure's words (re-identification key: memory only)


def _compact(s):
    return re.sub(r"[^a-z0-9]", "", str(s or "").lower())


def azure_surrogate_chunks(text):
    """[(base_offset, original_chunk, surrogate_chunk)] from the Surrogate operation."""
    out = []
    for base, chunk in _split_chunks(text, AZURE_MAX_CHARS):
        if not chunk.strip():
            out.append((base, chunk, chunk))
            continue
        result = call_azure_deidentification(chunk, "Surrogate")
        surrogate = result.get("outputText") or result.get("output_text")
        if surrogate is None:
            raise RuntimeError(f"No outputText in the Azure Surrogate response. Keys returned: {list(result)}")
        out.append((base, chunk, surrogate))
    return out


def _distribute(new_tokens, k):
    """Spread Azure's tokens over k original words (extras join the last word, missing ones leave a word empty)."""
    if k <= 0:
        return []
    if len(new_tokens) >= k:
        return new_tokens[:k - 1] + [" ".join(new_tokens[k - 1:])]
    return new_tokens + [""] * (k - len(new_tokens))


_EDGE = r"""[^\w]*"""

def _shape_like(piece, orig):
    """Azure's piece with the original word's leading/trailing punctuation and letter case."""
    if not piece:
        return ""
    lead = re.match(r"^" + _EDGE, orig).group()
    trail = re.search(_EDGE + r"$", orig).group() if orig.strip(".,:;()[]{}'\"-/") else ""
    core = re.sub(r"^" + _EDGE, "", piece)
    core = re.sub(_EDGE + r"$", "", core) or piece
    new = lead + core + trail
    letters = [c for c in orig if c.isalpha()]
    if MATCH_CASE and letters and any(c.isalpha() for c in new):
        if all(c.isupper() for c in letters):
            new = new.upper()
        elif all(c.islower() for c in letters):
            new = new.lower()
    return new


def _align_view(text, start_to_word, view, rows, counts, parts):
    """Azure Surrogate on one text. Returns {word_id: new text} for every OCR word Azure changed.
    start_to_word maps the position where a VALUE word starts in `text` to its word id; label words of the
    label: value text have no id, so Azure's changes to them are never applied."""
    out = {}
    segments = []
    for base, chunk, surrogate in azure_surrogate_chunks(text):
        parts.append(surrogate)
        c_lines, s_lines = chunk.split("\n"), surrogate.split("\n")
        if len(c_lines) == len(s_lines):              # Azure kept the line breaks: align line by line (safest)
            off = 0
            for cl, sl in zip(c_lines, s_lines):
                segments.append((base + off, cl, sl))
                off += len(cl) + 1
        else:                                          # otherwise align the whole chunk at once
            segments.append((base, chunk, surrogate))

    for seg_base, seg, sur in segments:
        src = [(m.group(), start_to_word.get(seg_base + m.start())) for m in re.finditer(r"\S+", seg)]
        dst = sur.split()
        if [tok for tok, _ in src] == dst:
            continue                                                 # line unchanged
        matcher = SequenceMatcher(None, [tok for tok, _ in src], dst, autojunk=False)
        for op, i1, i2, j1, j2 in matcher.get_opcodes():
            if op == "equal":
                continue
            if op == "insert":                                       # Azure added words: nowhere to draw them
                counts["inserts"] += 1
                continue
            part, new = src[i1:i2], dst[j1:j2]
            if all(wid for _, wid in part):                          # only OCR value words: one block
                groups = [([wid for _, wid in part], [tok for tok, _ in part], new)]
            elif len(part) == len(new):                              # label + value words mixed: value words 1:1
                groups = [([wid], [tok], [n]) for (tok, wid), n in zip(part, new) if wid]
            else:
                if any(wid for _, wid in part):
                    counts["unaligned"] += 1                         # cannot tell which new word belongs where
                continue
            for wids, orig, new_words in groups:
                key = _compact(" ".join(orig))                       # same value seen before -> same fake
                if REUSE_SURROGATES and key:
                    if key in SURROGATE_MEMO and SURROGATE_MEMO[key] != new_words:
                        new_words = SURROGATE_MEMO[key]
                        counts["reused"] += len(wids)
                    else:
                        SURROGATE_MEMO.setdefault(key, new_words)
                for wid, tok, piece in zip(wids, orig, _distribute(new_words, len(wids))):
                    out[wid] = _shape_like(piece, tok)
                rows.append((view, " ".join(orig), " ".join(out[w] for w in wids), wids))
    return out



_DATE_LIKE = re.compile(r"^\(?\d{1,4}[/.-]\d{1,2}([/.-]\d{2,4})?\)?$")
_COMPACT_DATE_FORMATS = {8: ("%m%d%Y",), 6: ("%m%d%y",)}
_SEP_DATE_FORMATS = ("%m/%d/%Y", "%m/%d/%y", "%m-%d-%Y", "%m-%d-%y", "%Y-%m-%d", "%m.%d.%Y", "%m %d %Y", "%m %d %y")

def is_date_text(s, compact6=True):
    """True if s is a complete, valid date: 05311968 / 053168 (no separators), 05/31/1968, 05 31 1968 ..."""
    from datetime import datetime as _dt
    s = str(s or "").strip().strip("().,:;")
    fmts = _COMPACT_DATE_FORMATS.get(len(s), ()) if s.isdigit() else _SEP_DATE_FORMATS
    if s.isdigit() and len(s) == 6 and not compact6:
        return False
    for fmt in fmts:
        try:
            d = _dt.strptime(s, fmt)
        except ValueError:
            continue
        if 1900 <= d.year <= 2100:
            return True
    return False


def _guess_kind(text):
    """Kind for a word only Azure flagged (Surrogate gives no category): used for propagation and the fallback fake."""
    if "@" in text and re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text):
        return "email"
    a = re.sub(r"[^A-Za-z0-9/.-]", "", text)
    if _DATE_LIKE.match(a) or (a.isdigit() and len(a) == 8 and is_date_text(a)):
        return "date"
    if any(ch.isdigit() for ch in a):
        return "id"
    return "name"


def azure_surrogate_detect(full_text, offset_map, ctx_text, ctx_map):
    """Azure Surrogate on the page text and the label: value text.
    Returns {"page": {word_id: new_text}, "ctx": {...}, "rows": [...], "page_out": str, "ctx_out": str}."""
    rows, counts, page_parts, ctx_parts = [], {"inserts": 0, "unaligned": 0, "reused": 0}, [], []
    page = _align_view(full_text, {e['start']: e['word_id'] for e in offset_map}, "page", rows, counts, page_parts)
    ctx = {}
    if ctx_text.strip() and ctx_map:
        ctx = _align_view(ctx_text, {e['start']: e['word_id'] for e in ctx_map}, "label:value", rows, counts, ctx_parts)
    return {"page": page, "ctx": ctx, "rows": rows, "counts": counts,
            "page_out": "".join(page_parts), "ctx_out": "".join(ctx_parts)}


def debug_texts(full_text, ctx_text, azure_result, offset_map):
    """Stage 3t: what was sent to Azure and what Azure returned, for both texts."""
    _dbg_title("STAGE 3t-1 - Page text (full_text) sent to Azure Surrogate")
    print(full_text)
    _dbg_title("STAGE 3t-2 - Azure Surrogate output for the page text")
    print(azure_result.get("page_out", ""))
    if ctx_text.strip():
        _dbg_title("STAGE 3t-3 - GPT label: value text (ctx_text) sent to Azure Surrogate")
        print(ctx_text)
        _dbg_title("STAGE 3t-4 - Azure Surrogate output for the label: value text")
        print(azure_result.get("ctx_out", ""))



# ---------------------------------------------------------------- settings
MISSED_PHI_ACTION = "erase"   # GPT says PHI, Azure changed it in neither text: "erase" (blank it) or "keep"
FIX_INVALID_DATES = True      # Azure's fake for a date is not a valid date (e.g. 05311968 -> 48213907): shift the real date
EMAIL_FAKES = "realistic"     # "realistic": names + common provider (c.smith85@x.org -> d.wilson42@outlook.com); "azure": Azure's

_KEY = (os.environ.get("DEID_SURROGATE_KEY") or os.urandom(16).hex()).encode()
def _rng(*parts):
    digest = hmac.new(_KEY, "|".join(parts).encode(), hashlib.sha256).digest()
    return random.Random(int.from_bytes(digest[:8], "big"))
DATE_SHIFT_DAYS = -(1 + _rng("date-shift").randrange(364))     # one fixed shift for every date (set DEID_SURROGATE_KEY in .env)

_SHIFT_FORMATS = ["%m/%d/%Y", "%m/%d/%y", "%m-%d-%Y", "%m-%d-%y", "%Y-%m-%d", "%m.%d.%Y", "%m %d %Y", "%m %d %y",
                  "%m%d%Y", "%m%d%y"]

def shift_date(s):
    """The date shifted by DATE_SHIFT_DAYS, in the same format (05311968 -> 01261968). None if s is not a date."""
    s = s.strip()
    for fmt in _SHIFT_FORMATS:
        if (fmt == "%m%d%Y" and len(s) != 8) or (fmt == "%m%d%y" and len(s) != 6):
            continue
        try:
            d = datetime.strptime(s, fmt)
        except ValueError:
            continue
        if 1900 <= d.year <= 2100:
            return (d + timedelta(days=DATE_SHIFT_DAYS)).strftime(fmt)
    return None


_FIRST = "amy ann ben dan eva ian jim kim max sam tom zoe adam alan anna carl dale ella emma eric gary jack jane john judy kate lisa luke mark mary neil nina noah paul ruth ryan sara sean tina tony alice brian carol chloe david diana emily frank grace helen henry irene jason karen kevin laura linda maria nancy oscar peter ralph robin sarah simon susan terry andrew austin brenda daniel dennis donald edward george hannah joseph julian lauren marcus monica nathan olivia philip rachel robert sandra steven teresa victor walter".split()
_LAST = "lee kim ray cox day fox hall king long ward west bell cook dean ford gray hart hill hunt lane nash reed ross shaw wade webb wood adams allen baker black brown burns clark davis evans floyd grant green hayes james jones kelly lewis mason moore myers perry price reyes scott stone wells white young archer bailey barnes bishop bryant butler carter cooper dalton fisher foster gibson gordon graham harris hudson jordan keller lawson martin miller morgan murphy nelson palmer parker porter powell rogers taylor thomas turner walker warren watson wilson".split()
_EMAIL_DOMAINS = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "aol.com", "icloud.com", "mail.com", "live.com"]
EMAIL_RX = re.compile(r"^([A-Za-z0-9._%+-]+)@([A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*)\.([A-Za-z]{2,})$")

def realistic_email(core):
    """c.smith85@stjoseph.org -> d.wilson42@outlook.com : real names, same separators, digits replaced, common provider."""
    m = EMAIL_RX.match(core.strip())
    if not m:
        return None
    rng = _rng("email", core.strip().lower())
    runs = re.findall(r"[A-Za-z]+|\d+|[^A-Za-z\d]+", m.group(1))
    letter_runs = [r for r in runs if r[0].isalpha()]
    out, k = [], 0
    for run in runs:
        if run[0].isalpha():
            pool = _LAST if (k == len(letter_runs) - 1 and len(letter_runs) > 1) or len(run) > 3 else _FIRST
            close = sorted(pool, key=lambda n: (abs(len(n) - len(run)), n))[:8]
            out.append(rng.choice([c for c in "abcdefghjklmnprstw" if c != run.lower()]) if len(run) == 1
                       else rng.choice([n for n in close if n != run.lower()] or close))
            k += 1
        elif run[0].isdigit():
            out.append("".join(rng.choice("0123456789") for _ in run))
        else:
            out.append(run)
    fake = "".join(out) + "@" + rng.choice(_EMAIL_DOMAINS)
    letters = [c for c in core if c.isalpha()]
    return fake.upper() if letters and all(c.isupper() for c in letters) else fake


def _digits(s):
    return re.sub(r"\D", "", s)


# ---------------------------------------------------------------- the decision
def decide_changes(full_text, offset_map, ctx_text, ctx_map, kv_pairs, vlm_hits, protected_ids):
    """Sets w['new_text'] on every word to change. Returns (flags, entities, stats)."""
    word_by_id = {w['word_id']: w for w in offset_map}
    for w in offset_map:
        w.pop("new_text", None)                                    # clean state when a cell is re-run
    azure = azure_surrogate_detect(full_text, offset_map, ctx_text, ctx_map)

    gpt_phi_ids = {wid for h in (vlm_hits or []) for wid in h["word_ids"]}
    gpt_nonphi_ids = {wid for p in (kv_pairs or []) for wid in p['value_word_ids']} - gpt_phi_ids

    # 1. Azure's changes: page text first, the label: value text only for words the page text left unchanged
    new, source = {}, {}
    for view, changes in (("azure_page", azure["page"]), ("azure_label_value", azure["ctx"])):
        for wid, txt in changes.items():
            if wid not in new and txt != word_by_id[wid]['text']:
                new[wid], source[wid] = txt, view

    # 2. never changed: printed form text / field names, and values GPT says are not PHI
    kept_protected, kept_nonphi, decision_kept = [], [], {}
    for wid in list(new):
        w = word_by_id[wid]
        if wid in protected_ids:
            kept_protected.append((w['text'], new.pop(wid))); decision_kept[wid] = "kept: printed form text / field name"
        elif wid in gpt_nonphi_ids:
            kept_nonphi.append((w['text'], new.pop(wid))); decision_kept[wid] = "kept: GPT says not PHI"

    # 3. GPT PHI of which Azure changed no word in either text
    missed = []
    for h in (vlm_hits or []):
        ids = [wid for wid in h["word_ids"] if wid in word_by_id]
        if any(wid in new for wid in ids):
            continue
        for wid, span in zip(h["word_ids"], h["spans"]):
            w = word_by_id.get(wid)
            if w is None or wid in protected_ids:
                continue
            missed.append((h["kind"], w['text']))
            if MISSED_PHI_ACTION == "erase":
                lo, hi = span if span else (0, len(w['text']))
                new[wid], source[wid] = w['text'][:lo] + w['text'][hi:], "erased_gpt_phi"

    # group the changed words into values (consecutive words on one line)
    entities = []
    for w in offset_map:
        if w['word_id'] not in new:
            continue
        last = entities[-1] if entities else None
        if last and last["line_idx"] == w['line_idx'] and last["words"][-1]['idx'] == w['idx'] - 1:
            last["words"].append(w)
        else:
            entities.append({"kind": "phi", "line_idx": w['line_idx'], "words": [w], "conf": 1.0})

    # 4. format fixes, per value
    dates_fixed = emails_fixed = 0
    for ent in entities:
        ids = [w['word_id'] for w in ent["words"]]
        if any(source.get(i) == "erased_gpt_phi" for i in ids):
            continue
        original = " ".join(w['text'] for w in ent["words"])
        fake = " ".join(new[i] for i in ids)
        plain = original.strip("().,:;")
        if FIX_INVALID_DATES and is_date_text(plain):
            ok = is_date_text(fake.strip("().,:;")) and len(_digits(fake)) == len(_digits(original))
            if not ok:
                shifted = shift_date(plain)
                if shifted:
                    parts = shifted.split(" ")
                    pieces = parts if len(parts) == len(ids) else [shifted] + [""] * (len(ids) - 1)
                    for i, piece in zip(ids, pieces):
                        new[i] = _shape_like(piece, word_by_id[i]['text'])
                        source[i] = "date_shifted"
                    dates_fixed += 1
        for i in ids:
            t = word_by_id[i]['text']
            core = t.strip("()<>[],;:")
            if "@" in core and EMAIL_RX.match(core) and (EMAIL_FAKES == "realistic" or not EMAIL_RX.match(new[i].strip("()<>[],;:"))):
                e = realistic_email(core)
                if e:
                    new[i] = t.replace(core, e)
                    source[i] = "email_realistic"
                    emails_fixed += 1

    # 5. flags + entities in the shape the rendering / verification code expects
    flags = {}
    for wid, txt in new.items():
        w = word_by_id[wid]
        if txt == w['text']:
            continue
        w["new_text"] = txt
        flags[wid] = {"kind": "phi", "prio": 2, "conf": 1.0, "mask": [True] * len(w['text']), "sources": {source[wid]}}
    for ent in entities:
        ent["words"] = [w for w in ent["words"] if w['word_id'] in flags]   # drop words whose fake equals the original
    entities = [e for e in entities if e["words"]]
    for ent in entities:
        ent["original"] = " ".join(w['text'] for w in ent["words"])
        ent["surrogate"] = " ".join(w['new_text'] for w in ent["words"])
        ent["sources"] = sorted(set().union(*(flags[w['word_id']]["sources"] for w in ent["words"])))

    rows = [(e["original"], e["surrogate"], ",".join(e["sources"])) for e in entities]
    stats = {"changed_words": len(flags), "changed_values": len(entities),
             "by_page_text": sum(1 for f in flags.values() if "azure_page" in f["sources"]),
             "by_label_value_text": sum(1 for f in flags.values() if "azure_label_value" in f["sources"]),
             "kept_gpt_not_phi": len(kept_nonphi), "kept_protected": len(kept_protected),
             "gpt_phi_not_changed_by_azure": len(missed), "dates_fixed": dates_fixed, "emails_realistic": emails_fixed,
             "azure_inserts_skipped": azure["counts"]["inserts"], "unaligned_blocks_skipped": azure["counts"]["unaligned"],
             "reused_from_memo_words": azure["counts"]["reused"],
             "rows": rows, "kept_nonphi_rows": kept_nonphi, "kept_protected_rows": kept_protected, "missed_rows": missed,
             "azure": azure, "decision_kept": decision_kept, "_sources": {wid: f["sources"] for wid, f in flags.items()},
             "gpt_phi_ids": gpt_phi_ids, "gpt_nonphi_ids": gpt_nonphi_ids}
    return flags, entities, stats


def final_page_text(offset_map):
    by_line = {}
    for w in offset_map:
        by_line.setdefault(w['line_idx'], []).append(w.get('new_text', w['text']))
    return "\n".join(" ".join(t for t in ws if t) for _, ws in sorted(by_line.items()))


SAVE_TEXT_FILES = True   # <name>_changes.txt, <name>_surrogated.txt, <name>_gpt.json next to the output image

def save_text_files(stats, offset_map, vlm_stats, ctx_text, out_dir, stem):
    """_changes.txt (original -> fake, kept and erased values), _surrogated.txt (final text), _gpt.json (GPT's answer).
    _changes.txt and _gpt.json hold REAL PHI: keep them as protected as the original images."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    w = max([len(o) for o, _, _ in stats["rows"]] + [8])
    out = [f"Changes for {stem}", "=" * 80, "", f"{'ORIGINAL':<{w}}  ->  FAKE   (from)", "-" * (w + 50)]
    out += [f"{o:<{w}}  ->  {n}   ({s})" for o, n, s in stats["rows"]]
    out += ["", f"Kept: GPT says not PHI, Azure wanted to change it ({len(stats['kept_nonphi_rows'])})", "-" * 60]
    out += [f"{o}  (Azure: {n})" for o, n in stats["kept_nonphi_rows"]]
    out += ["", f"Kept: printed form text / field name ({len(stats['kept_protected_rows'])})", "-" * 60]
    out += [f"{o}  (Azure: {n})" for o, n in stats["kept_protected_rows"]]
    out += ["", f"GPT says PHI, Azure changed it in neither text ({MISSED_PHI_ACTION}) ({len(stats['missed_rows'])})", "-" * 60]
    out += [f"{k}: {t}" for k, t in stats["missed_rows"]]
    paths = [out_dir / f"{stem}_changes.txt", out_dir / f"{stem}_surrogated.txt", out_dir / f"{stem}_gpt.json"]
    paths[0].write_text("\n".join(out) + "\n", encoding="utf-8")
    paths[1].write_text(final_page_text(offset_map) + "\n", encoding="utf-8")
    paths[2].write_text(json.dumps({"error": vlm_stats.get("vlm_error", ""), "answer": vlm_stats.get("vlm_raw"),
                                    "fields_matched": vlm_stats.get("field_rows", []),
                                    "free_text_matched": vlm_stats.get("free_rows", []),
                                    "label_value_text_sent_to_azure": ctx_text}, indent=1, ensure_ascii=False, default=str),
                        encoding="utf-8")
    return paths


SAVE_EXCEL_REPORT = True    # <name>_deid_report.xlsx next to the output image (+ batch_changes.xlsx for a batch)
BATCH_WORD_ROWS = []        # every page's "Word decisions" rows, for batch_changes.xlsx

_DECISION_TEXT = {"azure_page": "changed: Azure (page text)", "azure_label_value": "changed: Azure (label: value text)",
                  "erased_gpt_phi": "erased: GPT says PHI, Azure did not change it",
                  "date_shifted": "changed: date shifted (Azure's fake was not a valid date)",
                  "email_realistic": "changed: realistic e-mail"}


def save_deid_texts(full_text, ctx_text, stats, out_dir, stem):
    """The two texts sent to Azure and Azure's two answers, as text files."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    az = stats["azure"]
    files = {f"{stem}_ocr_text_sent.txt": full_text, f"{stem}_ocr_text_deid.txt": az.get("page_out", ""),
             f"{stem}_gpt_text_sent.txt": ctx_text, f"{stem}_gpt_text_deid.txt": az.get("ctx_out", "")}
    for name, text in files.items():
        (out_dir / name).write_text(text or "", encoding="utf-8")
    return [out_dir / n for n in files]


def save_excel_report(image_path, full_text, offset_map, ctx_text, kv_pairs, vlm_stats, stats, out_dir, stem):
    """<stem>_deid_report.xlsx: Summary | OCR text | GPT text | Word decisions. Holds REAL PHI: keep it protected."""
    import pandas as pd
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    az = stats["azure"]
    word_by_id = {w['word_id']: w for w in offset_map}

    # OCR text, line by line
    sent_l, out_l = full_text.split("\n"), (az.get("page_out", "") or "").split("\n")
    if len(out_l) != len(sent_l):
        out_l = out_l + [""] * (len(sent_l) - len(out_l))
    ocr_rows = [{"line": n, "sent_to_azure": s, "azure_output": o, "changed_by_azure": s != o}
                for n, (s, o) in enumerate(zip(sent_l, out_l)) if s.strip() or o.strip()]

    # GPT text: one row per GPT field (matched fields are the lines of the label: value text, in order)
    ctx_sent, ctx_out = ctx_text.split("\n"), (az.get("ctx_out", "") or "").split("\n")
    gpt_rows, k = [], 0
    for r in vlm_stats.get("field_rows", []):
        row = {"gpt_label": r["label"], "gpt_value": r["value"], "category": r["category"], "is_phi": r["is_phi"],
               "found_in_ocr": r["matched"], "sent_to_azure": "", "azure_output": "", "changed_by_azure": ""}
        if str(r["matched"]).startswith("yes") and k < len(kv_pairs):
            row["sent_to_azure"] = ctx_sent[k] if k < len(ctx_sent) else ""
            row["azure_output"] = ctx_out[k] if k < len(ctx_out) else ""
            row["changed_by_azure"] = row["sent_to_azure"] != row["azure_output"]
            k += 1
        gpt_rows.append(row)
    for r in vlm_stats.get("free_rows", []):
        gpt_rows.append({"gpt_label": "(free-text PHI)", "gpt_value": r["value"], "category": r["category"], "is_phi": True,
                         "found_in_ocr": r["matched"], "sent_to_azure": "(not sent: no label)", "azure_output": "",
                         "changed_by_azure": ""})

    # Word decisions: every word Azure changed in either text, or that is changed / kept on the image
    ids = set(az.get("page", {})) | set(az.get("ctx", {})) | {w['word_id'] for w in offset_map if "new_text" in w} \
          | set(stats.get("decision_kept", {}))
    word_rows = []
    for w in offset_map:
        wid = w['word_id']
        if wid not in ids:
            continue
        drawn = w.get("new_text", w['text'])
        src = next(iter(flags_sources(wid, stats, w)), "")
        decision = stats["decision_kept"].get(wid) or _DECISION_TEXT.get(src) or (
            "unchanged" if drawn == w['text'] else "changed")
        word_rows.append({"image": Path(image_path).name, "line": w['line_idx'], "word_id": wid, "original": w['text'],
                          "azure_page_text": az.get("page", {}).get(wid, ""),
                          "azure_label_value_text": az.get("ctx", {}).get(wid, ""),
                          "gpt_says": "PHI" if wid in stats.get("gpt_phi_ids", ()) else
                                      "not PHI" if wid in stats.get("gpt_nonphi_ids", ()) else "",
                          "drawn_on_image": drawn, "decision": decision, "handwritten": w['handwriting']})
    BATCH_WORD_ROWS.extend(word_rows)

    summary = {k2: v for k2, v in stats.items() if not k2.endswith(("_rows", "_ids"))
               and k2 not in ("rows", "azure", "decision_kept", "_sources")}
    summary.update({"image": str(image_path), "gpt_fields": vlm_stats.get("vlm_fields", 0),
                    "gpt_fields_found_in_ocr": vlm_stats.get("vlm_fields_matched", 0),
                    "gpt_error": vlm_stats.get("vlm_error", ""),
                    "deid_calls": AZURE_USAGE["calls"], "deid_input_bytes": AZURE_USAGE["input_bytes"]})
    path = out_dir / f"{stem}_deid_report.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        pd.DataFrame([{"item": a, "value": b} for a, b in summary.items()]).to_excel(xw, sheet_name="Summary", index=False)
        pd.DataFrame(ocr_rows or [{}]).to_excel(xw, sheet_name="OCR text", index=False)
        pd.DataFrame(gpt_rows or [{}]).to_excel(xw, sheet_name="GPT text", index=False)
        pd.DataFrame(word_rows or [{}]).to_excel(xw, sheet_name="Word decisions", index=False)
        for ws in xw.book.worksheets:                                     # readable column widths
            for col in ws.columns:
                width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
                ws.column_dimensions[col[0].column_letter].width = min(60, max(10, width + 2))
            ws.freeze_panes = "A2"
    return path


def flags_sources(wid, stats, w):
    """The source of the fake for one word (azure_page, azure_label_value, erased_gpt_phi, date_shifted, ...)."""
    return stats.get("_sources", {}).get(wid, set())


def save_batch_excel(path):
    """All pages' word decisions in one workbook (batch_changes.xlsx)."""
    import pandas as pd
    if BATCH_WORD_ROWS:
        with pd.ExcelWriter(path, engine="openpyxl") as xw:
            pd.DataFrame(BATCH_WORD_ROWS).to_excel(xw, sheet_name="Word decisions", index=False)
            ws = xw.book.worksheets[0]; ws.freeze_panes = "A2"
            for col in ws.columns:
                width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
                ws.column_dimensions[col[0].column_letter].width = min(60, max(10, width + 2))
    return path


def debug_decisions(stats):
    _dbg_title("STAGE 3 - Values changed: original -> fake (and where the fake came from)")
    _dbg_rows(stats["rows"], ("original", "fake", "from"))
    _dbg_title("STAGE 3b - Kept: GPT says NOT PHI, Azure wanted to change it")
    _dbg_rows(stats["kept_nonphi_rows"], ("word", "azure_wanted"))
    _dbg_title("STAGE 3c - Kept: printed form text / field name, Azure wanted to change it")
    _dbg_rows(stats["kept_protected_rows"], ("word", "azure_wanted"))
    _dbg_title(f"STAGE 3d - GPT says PHI, Azure changed it in neither text ({MISSED_PHI_ACTION})")
    _dbg_rows(stats["missed_rows"], ("gpt_kind", "word"))


# ---------------------------------------------------------------- run for this page
from pathlib import Path
SURROGATE_MEMO.clear()
flags, entities, change_stats = decide_changes(full_text, offset_map, ctx_text, ctx_map, kv_pairs, vlm_hits, vlm_label_ids)
print("Changes:", {k: v for k, v in change_stats.items()
                   if not k.endswith(("_rows", "_ids")) and k not in ("rows", "azure", "decision_kept", "_sources")})
print(f"Azure DeID usage so far: {AZURE_USAGE}")
if DEBUG:
    debug_texts(full_text, ctx_text, change_stats["azure"], offset_map)
    debug_decisions(change_stats)

## 13. Replace on the Image

**Fake values of any length** (Azure's fakes keep no fixed length), in this order:
1. a longer fake grows into free space to its right, up to `MAX_EXTRA_WIDTH` (100 %) wider, never past the next word or a vertical form line;
2. if unchanged words after it are in the way, they are moved along with it, up to a form line (`CARRY_WORDS`);
3. still too long: condensed down to `MIN_SQUEEZE` (75 %), then drawn smaller down to `MIN_FONT_SCALE` (80 %), then condensed as far as needed;
4. a shorter fake starts where the original started.

**Handwriting fonts:** handwritten words (Textract `HANDWRITING`) are drawn in the installed handwriting font that best reproduces the original strokes (same fit test as printed text). `HANDWRITING_FONT_FILES` lists about 50 candidates: Windows (Ink Free, Segoe Script, Segoe Print, Comic Sans), Office (Bradley Hand, Lucida Handwriting, Kristen, Freestyle Script, Mistral, Pristina, Juice), and free Google Fonts, print-style (Caveat, Patrick Hand, Kalam, Indie Flower, Handlee, Architects Daughter, Shadows Into Light, Gloria Hallelujah, Covered By Your Grace, Just Another Hand, Reenie Beanie, Gochi Hand, Neucha, Nanum Pen Script…) and cursive / signature style (Dancing Script, Homemade Apple, Satisfy, Sacramento, Allura, Great Vibes, Mr Dafoe…). Fonts not installed are skipped. Put any other fonts in a folder and add it to `HANDWRITING_FONT_DIRS`: every `.ttf` / `.otf` in it is used. More fonts = a better match but a little slower per handwritten word.

Stage 5 and the summary show `condensed`, `smaller font`, `moved` and `handwriting font` per word. Font and size fitting, erasing, colour and handwriting fonts are unchanged.

**Rewritten.** Font and size are fitted on the original pixels, only the ink is erased, and each new word is drawn at the original position and baseline. Unreliable words get a solid box instead.

In [ ]:
# Put the fake values on the image. Principle: "fit on the original, then swap the text".
# For every PHI word we know both the original text (from OCR) and its pixels, so:
#   1. FONT  - render the ORIGINAL text in each candidate font and score it against
#              the real pixels; keep the best one per line (steadied by other text of
#              the same size), so mixed-font documents work.
#   2. SIZE  - the size at which the original text is exactly as wide as its ink;
#              one size per line, so neighbouring words never differ.
#   3. ERASE - only the ink pixels inside the word box, never the form lines, filled
#              with real background pixels sampled next to the word (works on white,
#              grey and coloured forms).
#   4. DRAW  - each new word at the original word's x position and baseline, in the
#              original ink colour, slightly blurred like a scan.
# Handwritten words use a handwriting font. Fakes longer than the original grow into free
# space, push unchanged words along, then are condensed / drawn smaller (see the section text).
import io
import numpy as np
from collections import Counter as _Counter
from functools import lru_cache
from PIL import Image, ImageDraw, ImageFont, ImageFilter

# ---------------------------------------------------------------- settings
MIN_OCR_CONF = 70.0     # below this the OCR text is not trusted -> solid box
MAX_SKEW_DEG = 3.0      # words rotated more than this -> solid box
ROTATE_MIN_DEG = 0.7    # words rotated more than this are drawn rotated
CALIB_WORDS = 60        # extra non-PHI words sampled to learn the fonts on the page
FONT_PEER_WEIGHT = 0.15 # how strongly same-size text elsewhere settles a near-tie between fonts
BLUR_RADIUS = 0.45      # softening of the drawn text (0 = crisp)
BOX_COLOR = (255, 0, 0)   # colour of the fail-safe box
EXTRA_FONT_PATHS = []   # add full paths of any other .ttf/.otf you want considered
KEEP_LOW_CONFIDENCE = False   # False: low-confidence PHI words are replaced too (True left them visible)

# Handwritten words (Textract TextType = HANDWRITING) are redrawn in a handwriting-style font.
# The best of these fonts is chosen per word by the same fit test as printed text.
# Fonts that are not installed are skipped; with none found, handwriting falls back to the line's printed font.
USE_HANDWRITING_FONT = True
HANDWRITING_FONT_FILES = [
    # Windows (usually installed)
    "Inkfree.ttf", "segoesc.ttf", "segoescb.ttf", "segoepr.ttf", "segoeprb.ttf", "comic.ttf", "comici.ttf",
    # Microsoft Office (installed with Office)
    "BRADHITC.TTF", "LHANDW.TTF", "ITCKRIST.TTF", "FREESCPT.TTF", "MISTRAL.TTF", "PRISTINA.TTF", "JUICE___.TTF",
    "TEMPSITC.TTF", "PAPYRUS.TTF",
    # Google Fonts (free download) - print-style handwriting
    "Caveat-Regular.ttf", "Caveat-VariableFont_wght.ttf", "PatrickHand-Regular.ttf", "PatrickHandSC-Regular.ttf",
    "Kalam-Regular.ttf", "Kalam-Light.ttf", "Kalam-Bold.ttf", "IndieFlower-Regular.ttf", "Handlee-Regular.ttf",
    "ArchitectsDaughter-Regular.ttf", "ShadowsIntoLight-Regular.ttf", "ShadowsIntoLightTwo-Regular.ttf",
    "GloriaHallelujah-Regular.ttf", "CoveredByYourGrace-Regular.ttf", "JustAnotherHand-Regular.ttf",
    "ReenieBeanie-Regular.ttf", "GochiHand-Regular.ttf", "Neucha-Regular.ttf", "SchoolBell-Regular.ttf",
    "NothingYouCouldDo-Regular.ttf", "RockSalt-Regular.ttf", "NanumPenScript-Regular.ttf", "Sriracha-Regular.ttf",
    "Mali-Regular.ttf", "Itim-Regular.ttf", "ComingSoon-Regular.ttf", "SueEllenFrancisco-Regular.ttf",
    # Google Fonts (free download) - cursive / signature style
    "DancingScript-Regular.ttf", "DancingScript-VariableFont_wght.ttf", "HomemadeApple-Regular.ttf",
    "Satisfy-Regular.ttf", "Sacramento-Regular.ttf", "Allura-Regular.ttf", "GreatVibes-Regular.ttf",
    "MrDafoe-Regular.ttf", "LaBelleAurore-Regular.ttf", "DawningofaNewDay-Regular.ttf", "Yellowtail-Regular.ttf",
]
EXTRA_HANDWRITING_FONT_PATHS = []   # full paths of any other handwriting .ttf/.otf
HANDWRITING_FONT_DIRS = []          # folders of handwriting fonts: EVERY .ttf/.otf inside is used (any file name),
                                    # e.g. [r"C:\Users\RSahu\Documents\Azure_deidentification\handwriting_fonts"]
HANDWRITING_ERASE_GROW = 0.2        # erase this fraction of the word height around a handwritten word (loops, tails)

# Fake values that are LONGER than the original (Azure's fakes keep no fixed length), in this order:
#   1. grow into free space to the right (never past the next word or a vertical form line)
#   2. condense horizontally, down to MIN_SQUEEZE
#   3. draw in a smaller font, down to MIN_FONT_SCALE (then condense as much as needed)
# SHORTER fakes start where the original started; the rest of the box stays blank.
MAX_EXTRA_WIDTH = 1.0    # a fake may be up to 100 % wider than the original when the space to its right is free
CARRY_WORDS = True       # a longer fake may push the unchanged words after it (same line) along, up to a form line
MIN_SQUEEZE = 0.75       # condense down to 75 % of the natural width before making the font smaller
MIN_FONT_SCALE = 0.80    # font size may go down to 80 % of the original

FONT_DIRS = [r"C:\Windows\Fonts", os.path.expanduser(r"~\AppData\Local\Microsoft\Windows\Fonts"),
             "/usr/share/fonts", "/Library/Fonts", "/System/Library/Fonts/Supplemental"]
FONT_FILES = [  # typewriter, sans, narrow sans, serif, each regular + bold
    "cour.ttf", "courbd.ttf", "consola.ttf", "consolab.ttf", "lucon.ttf", "OCRAEXT.TTF",
    "arial.ttf", "arialbd.ttf", "arialn.ttf", "arialnb.ttf", "calibri.ttf", "calibrib.ttf",
    "verdana.ttf", "tahoma.ttf", "segoeui.ttf", "times.ttf", "timesbd.ttf",
    # metric-compatible free equivalents (Linux / macOS)
    "LiberationMono-Regular.ttf", "LiberationMono-Bold.ttf", "LiberationSans-Regular.ttf",
    "LiberationSans-Bold.ttf", "LiberationSansNarrow-Regular.ttf", "LiberationSerif-Regular.ttf",
    "LiberationSerif-Bold.ttf", "DejaVuSans.ttf", "DejaVuSansMono.ttf",
]

def find_fonts():
    wanted = {f.lower() for f in FONT_FILES}
    found = {}
    for d in FONT_DIRS:
        if not os.path.isdir(d):
            continue
        for root, _dirs, files in os.walk(d):
            for f in files:
                if f.lower() in wanted and f.lower() not in found:
                    found[f.lower()] = os.path.join(root, f)
    paths = [found[f.lower()] for f in FONT_FILES if f.lower() in found]
    paths += [p for p in EXTRA_FONT_PATHS if os.path.exists(p)]
    if not paths:
        raise RuntimeError("No candidate fonts found. Add font file paths to EXTRA_FONT_PATHS.")
    return paths

FONTS = find_fonts()
print(f"Candidate fonts found: {len(FONTS)} -> {[os.path.basename(p) for p in FONTS]}")

def find_handwriting_fonts():
    wanted = {f.lower() for f in HANDWRITING_FONT_FILES}
    found = {}
    for d in FONT_DIRS:
        if not os.path.isdir(d):
            continue
        for root, _dirs, files in os.walk(d):
            for f in files:
                if f.lower() in wanted and f.lower() not in found:
                    found[f.lower()] = os.path.join(root, f)
    paths = [found[f.lower()] for f in HANDWRITING_FONT_FILES if f.lower() in found]
    paths += [p for p in EXTRA_HANDWRITING_FONT_PATHS if os.path.exists(p)]
    for d in HANDWRITING_FONT_DIRS:                      # every font file in these folders, whatever its name
        if os.path.isdir(d):
            for root, _dirs, files in os.walk(d):
                for f in sorted(files):
                    if f.lower().endswith((".ttf", ".otf")):
                        paths.append(os.path.join(root, f))
    seen, unique = set(), []
    for p_ in paths:                                     # no duplicates (same file found twice)
        key = os.path.basename(p_).lower()
        if key not in seen:
            seen.add(key); unique.append(p_)
    return unique

HANDWRITING_FONTS = find_handwriting_fonts() if USE_HANDWRITING_FONT else []
print(f"Handwriting fonts found: {len(HANDWRITING_FONTS)} -> {[os.path.basename(p) for p in HANDWRITING_FONTS]}"
      + ("" if HANDWRITING_FONTS or not USE_HANDWRITING_FONT else "  (none: handwriting uses the printed font of its line)"))


# ---------------------------------------------------------------- text rendering
@lru_cache(maxsize=4096)
def _font(path, size):
    return ImageFont.truetype(path, size)

@lru_cache(maxsize=100000)
def render_text(text, path, size):
    """Render text; returns (alpha HxW uint8 cropped to the ink, ink_left - origin_x, ink_top - baseline_y)."""
    font = _font(path, size)
    l, t, r, b = font.getbbox(text, anchor="ls")
    l, t, r, b = int(np.floor(l)), int(np.floor(t)), int(np.ceil(r)), int(np.ceil(b))
    pad = 4
    w, h = (r - l) + 2 * pad, (b - t) + 2 * pad
    if w <= 2 * pad or h <= 2 * pad:
        return None
    im = Image.new("L", (w, h), 0)
    ImageDraw.Draw(im).text((pad - l, pad - t), text, font=font, fill=255, anchor="ls")
    a = np.asarray(im)
    ys, xs = np.where(a >= 96)
    if len(xs) == 0:
        return None
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    return a[y0:y1, x0:x1].copy(), int(x0 - (pad - l)), int(y0 - (pad - t))


# ---------------------------------------------------------------- reading the original pixels
def _otsu(values):
    hist = np.bincount(values.astype(np.uint8).ravel(), minlength=256).astype(float)
    total = hist.sum()
    cum, cum_mean = np.cumsum(hist), np.cumsum(hist * np.arange(256))
    w0, w1 = cum, total - cum
    with np.errstate(divide="ignore", invalid="ignore"):
        m0, m1 = cum_mean / w0, (cum_mean[-1] - cum_mean) / w1
        between = w0 * w1 * (m0 - m1) ** 2
    between = np.where(np.isfinite(between), between, -1.0)
    top = between.max()
    best = np.flatnonzero(between >= top - abs(top) * 1e-6)
    return int((best[0] + best[-1]) // 2)

def _dilate(mask):
    p = np.pad(mask, 1)
    out = np.zeros_like(mask)
    for dy in (0, 1, 2):
        for dx in (0, 1, 2):
            out |= p[dy:dy + mask.shape[0], dx:dx + mask.shape[1]]
    return out

def _runs(column):
    """[(start, end)] of consecutive True values in a 1-D boolean array."""
    padded = np.concatenate([[False], column, [False]])
    edges = np.flatnonzero(padded[1:] != padded[:-1])
    return list(zip(edges[::2], edges[1::2]))

def _find_form_lines(ink):
    """Mask of ruled lines crossing the crop. A form line enters at one edge of the padded
    crop and leaves at the opposite edge (also when the scan is skewed); text never does.
    Text that touches a line is kept: only the thin band of the line itself is marked."""
    h, w = ink.shape
    lines = np.zeros_like(ink)

    def scan(m, out):                      # horizontal-ish lines of m, written into out
        hh, ww = m.shape
        if ww < 12:
            return
        xs = np.arange(ww)
        for s0, e0 in _runs(m[:, 0]):
            for s1, e1 in _runs(m[:, -1]):
                y0, y1 = (s0 + e0 - 1) / 2.0, (s1 + e1 - 1) / 2.0
                if abs(y0 - y1) > max(2.0, 0.06 * ww) or abs((e0 - s0) - (e1 - s1)) > 2:
                    continue
                ys = np.rint(y0 + (y1 - y0) * xs / (ww - 1)).astype(int)
                hit = np.zeros(ww, bool)
                for dy in (-1, 0, 1):
                    hit |= m[np.clip(ys + dy, 0, hh - 1), xs]
                if hit.mean() < 0.92:
                    continue
                half = int(np.ceil(max(e0 - s0, e1 - s1) / 2.0)) + 1
                for dy in range(-half, half + 1):
                    yy = ys + dy
                    ok = (yy >= 0) & (yy < hh)
                    out[yy[ok], xs[ok]] = True

    scan(ink, lines)
    if h >= 8:
        scan(ink.T, lines.T)               # vertical lines
    return lines

def analyze_word(gray, w, pad=2):
    """Find the ink of one word. Only pixels inside the word's (possibly tilted) polygon
    count, and form lines crossing the crop are detected and excluded."""
    x0, y0, x1, y1 = w['px']
    X0, Y0 = max(0, x0 - pad), max(0, y0 - pad)
    X1, Y1 = min(gray.shape[1], x1 + pad), min(gray.shape[0], y1 + pad)
    crop = gray[Y0:Y1, X0:X1]
    if crop.size == 0:
        return None

    inside = np.ones(crop.shape, bool)
    if w.get('poly') and abs(w['angle']) > 0.2:      # tilted word: its box is a rotated rectangle
        cx = sum(p[0] for p in w['poly']) / len(w['poly'])
        cy = sum(p[1] for p in w['poly']) / len(w['poly'])
        pts = [(px + (pad if px >= cx else -pad) - X0, py + (pad if py >= cy else -pad) - Y0) for px, py in w['poly']]
        im = Image.new("L", (crop.shape[1], crop.shape[0]), 0)
        ImageDraw.Draw(im).polygon(pts, fill=255)
        inside = np.asarray(im) > 0

    ink = np.zeros(crop.shape, bool)
    lines = np.zeros(crop.shape, bool)
    if int(crop.max()) - int(crop.min()) >= 40:
        dark = crop <= _otsu(crop)
        lines = _find_form_lines(dark)
        ink = dark & ~lines & inside
    ys, xs = np.where(ink)
    inkbox = (X0 + xs.min(), Y0 + ys.min(), X0 + xs.max() + 1, Y0 + ys.max() + 1) if len(xs) else None
    return {"origin": (X0, Y0), "ink": ink, "lines": lines, "inside": inside, "inkbox": inkbox}

def _tight_mask(info, angle=0.0):
    """The word's ink cropped tight; a skewed word is straightened first so it can be
    compared with horizontally rendered text."""
    X0, Y0 = info["origin"]
    ix0, iy0, ix1, iy1 = info["inkbox"]
    tight = info["ink"][iy0 - Y0:iy1 - Y0, ix0 - X0:ix1 - X0]
    if abs(angle) > ROTATE_MIN_DEG:
        im = Image.fromarray(tight.astype(np.uint8) * 255).rotate(angle, resample=Image.BILINEAR, expand=True)
        a = np.asarray(im) >= 128
        ys, xs = np.where(a)
        if len(xs):
            tight = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    return tight

def _ink_size(w, info):
    """(width, height) of the word's ink, measured along the text direction."""
    if not info["inkbox"]:
        return w['px'][2] - w['px'][0], w['px'][3] - w['px'][1]
    tight = _tight_mask(info, w['angle'])
    return tight.shape[1], tight.shape[0]

def _soft(mask_u8):
    return np.asarray(Image.fromarray(mask_u8).filter(ImageFilter.GaussianBlur(0.6)), dtype=float) / 255.0


# ---------------------------------------------------------------- font + size fitting
_REF = 64

def fit_size(text, path, ink_w, ink_h):
    """Font size at which `text` is as wide (or, for very short text, as tall) as its ink."""
    ref = render_text(text, path, _REF)
    if ref is None:
        return None
    rh, rw = ref[0].shape
    by_width = len(text.strip()) >= 3
    guess = _REF * (ink_w / rw if by_width else ink_h / rh)
    guess = int(round(min(300, max(5, guess))))
    best, best_err = None, None
    for s in (guess - 1, guess, guess + 1):
        if s < 5:
            continue
        r = render_text(text, path, s)
        if r is None:
            continue
        err = abs(r[0].shape[1] - ink_w) if by_width else abs(r[0].shape[0] - ink_h)
        if best_err is None or err < best_err:
            best, best_err = s, err
    return best

def score_fit(text, path, size, tight):
    """0..1: how well `text` in this font/size reproduces the word's real ink."""
    r = render_text(text, path, size)
    if r is None:
        return 0.0
    H, W = tight.shape
    h, w = r[0].shape
    rendered = _soft(np.asarray(Image.fromarray(r[0]).resize((W, H), Image.BILINEAR)))
    target = _soft(tight.astype(np.uint8) * 255)
    dice = 2 * np.minimum(rendered, target).sum() / (rendered.sum() + target.sum() + 1e-6)
    # proportions must agree too (1 px of tolerance, since small text is quantised)
    shape_penalty = max(0.0, 1 - 2 * max(0, abs(h - H) - 1) / max(H, 1)) * \
                    max(0.0, 1 - 2 * max(0, abs(w - W) - 1) / max(W, 1))
    return float(dice * shape_penalty)

def _fittable(w):
    return (not w['handwriting'] and abs(w['angle']) <= MAX_SKEW_DEG and w['confidence'] >= 90
            and sum(ch.isalnum() for ch in w['text']) >= 3)

def choose_styles(gray, offset_map, phi_word_ids):
    """Returns (main_font, {line_idx: style}). Fits fonts on the ORIGINAL text of sample words.

    A line's font is the one that best reproduces its own words. Near-ties are settled by
    which font wins most often among other text of a similar size, so a line holding only
    a number does not flip between, say, regular and bold from one box to the next.
    """
    phi_lines = {w['line_idx'] for w in offset_map if w['word_id'] in phi_word_ids}
    in_lines = [w for w in offset_map if w['line_idx'] in phi_lines and _fittable(w)]
    others = [w for w in offset_map if w['line_idx'] not in phi_lines and _fittable(w)]
    step = max(1, len(others) // CALIB_WORDS)
    calib = in_lines + others[::step][:CALIB_WORDS]

    fits = {}          # word_id -> {font: (score, size)}
    size_class = {}    # word_id -> bucket of its best-fit size
    for w in calib:
        info = analyze_word(gray, w)
        if not info or not info["inkbox"]:
            continue
        tight = _tight_mask(info, w['angle'])
        per_font = {}
        for path in FONTS:
            size = fit_size(w['text'], path, tight.shape[1], tight.shape[0])
            if size:
                per_font[path] = (score_fit(w['text'], path, size, tight), size)
        if per_font:
            fits[w['word_id']] = per_font
            best = max(per_font, key=lambda f: per_font[f][0])
            size_class[w['word_id']] = int(round(np.log(per_font[best][1]) / np.log(1.15)))

    def mean_scores(word_ids):
        out = {}
        for path in FONTS:
            vals = [fits[i][path][0] for i in word_ids if path in fits[i]]
            if vals:
                out[path] = float(np.mean(vals))
        return out

    def make_style(members, font):
        sizes = [fits[w['word_id']][font][1] for w in members if font in fits[w['word_id']]]
        return {"font": font, "size": int(round(float(np.median(sizes)))),
                "cx": float(np.mean([(w['px'][0] + w['px'][2]) / 2 for w in members])),
                "cy": float(np.mean([(w['px'][1] + w['px'][3]) / 2 for w in members]))}

    by_line = {}
    for w in calib:
        if w['word_id'] in fits:
            by_line.setdefault(w['line_idx'], []).append(w)

    # Pass 1: lines with enough letters to tell fonts apart.
    line_styles, weak = {}, []
    for line_idx, members in by_line.items():
        ids = [w['word_id'] for w in members]
        own = mean_scores(ids)
        if sum(ch.isalpha() for w in members for ch in w['text']) < 5:
            weak.append((line_idx, members, own))
            continue
        cls = int(np.median([size_class[i] for i in ids]))
        peers = [i for i in fits if abs(size_class[i] - cls) <= 1]
        votes = _Counter(max(fits[i], key=lambda f: fits[i][f][0]) for i in peers)
        font = max(own, key=lambda f: own[f] + FONT_PEER_WEIGHT * votes.get(f, 0) / len(peers))
        line_styles[line_idx] = make_style(members, font)

    # Pass 2: lines that are mostly digits ("62704", "03/15/2024"). Digits look alike in many
    # fonts, so take the font of the nearest lettered line of the same size, provided it
    # also reproduces this line about as well as this line's own best guess.
    strong = dict(line_styles)
    for line_idx, members, own in weak:
        best = max(own, key=own.get)
        style = make_style(members, best)
        near = sorted(strong.values(), key=lambda st: abs(st["cy"] - style["cy"]) * 3 + abs(st["cx"] - style["cx"]))
        for st in near:
            if st["font"] in own and own[st["font"]] >= own[best] - 0.08:
                candidate = make_style(members, st["font"])
                if abs(candidate["size"] - st["size"]) <= 0.15 * st["size"]:
                    style = candidate
                    break
        line_styles[line_idx] = style

    used = [st["font"] for i, st in line_styles.items() if i in phi_lines] or [st["font"] for st in line_styles.values()]
    main_font = _Counter(used).most_common(1)[0][0] if used else FONTS[0]
    return main_font, line_styles

def style_for_word(w, info, doc_font, line_styles):
    """Font and size for one PHI word: its own line's style, else the nearest line whose
    style reproduces this word's ink height (short values like "04" cannot be fitted alone).
    Handwritten words: the handwriting font that best reproduces the original strokes."""
    if w['handwriting'] and HANDWRITING_FONTS and info and info["inkbox"]:
        tight = _tight_mask(info, w['angle'])
        best = None
        for path in HANDWRITING_FONTS:
            size = fit_size(w['text'], path, tight.shape[1], tight.shape[0])   # size from the real ink
            if size:
                score = score_fit(w['text'], path, size, tight)                # same Dice score as printed text
                if best is None or score > best[0]:
                    best = (score, path, size)
        if best:
            return best[1], best[2]
    x0, y0, x1, y1 = w['px']
    ink_w, ink_h = _ink_size(w, info)
    own = line_styles.get(w['line_idx'])
    if own:
        font, size = own["font"], own["size"]
        ref = render_text(w['text'], font, size)
        if ref is not None and len(w['text'].strip()) >= 3 and abs(ref[0].shape[1] - ink_w) > 0.25 * ink_w:
            size = fit_size(w['text'], font, ink_w, ink_h) or size      # this word is a different size
        return font, size
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    best, best_d = None, None
    for st in line_styles.values():
        ref = render_text(w['text'], st["font"], st["size"])
        if ref is None or abs(ref[0].shape[0] - ink_h) > 0.15 * ink_h:
            continue
        d = abs(st["cy"] - cy) * 3 + abs(st["cx"] - cx)                 # prefer lines just above/below
        if best_d is None or d < best_d:
            best, best_d = st, d
    if best:
        return best["font"], best["size"]
    return doc_font, fit_size(w['text'], doc_font, ink_w, ink_h) or max(6, int(0.8 * (y1 - y0)))


# ---------------------------------------------------------------- erase + draw
def erase_word(out, orig, info, box, rng, page_bg):
    """Remove the word's ink (not the form lines) and fill with nearby background pixels."""
    X0, Y0 = info["origin"]
    h, w = info["ink"].shape
    x0, y0, x1, y1 = box
    mask = _dilate(info["ink"]) 
    if not info['ink'].any():
        mask=np.ones_like(mask)                     # also take the grey anti-aliased edge
    allowed = np.zeros_like(mask)                    # stay inside the word box (+1 px; more for handwriting)
    g = max(1, int(round(HANDWRITING_ERASE_GROW * (y1 - y0)))) if info.get("handwriting") else 1
    allowed[max(0, y0 - g - Y0):max(0, y1 + g - Y0), max(0, x0 - g - X0):max(0, x1 + g - X0)] = True
    mask &= allowed & ~info["lines"] & info["inside"]
    if not mask.any():
        return
    background = ~_dilate(_dilate(info["ink"])) & ~_dilate(info["lines"])
    region_orig = orig[Y0:Y0 + h, X0:X0 + w]
    samples = region_orig[background]
    if len(samples) < 8:
        samples = page_bg
    out[Y0:Y0 + h, X0:X0 + w][mask] = samples[rng.integers(0, len(samples), int(mask.sum()))]

def ink_colour(orig, info, ref):
    """Colour that makes the redrawn text carry the same amount of ink as the original.
    (A plain 'darkest pixel' is too dark, a median too light: thin scanned strokes are grey.)"""
    X0, Y0 = info["origin"]
    h, w = info["ink"].shape
    region = orig[Y0:Y0 + h, X0:X0 + w].astype(float)
    near = _dilate(info["ink"])
    background = ~_dilate(near) & ~_dilate(info["lines"])
    if not info["ink"].any() or background.sum() < 8:
        return np.array([0.0, 0.0, 0.0]), None
    bg = np.median(region[background], axis=0)
    near &= ~info["lines"]
    if ref is None:
        return np.median(region[info["ink"]], axis=0), bg
    mass = np.clip(bg - region[near], 0, None).sum(axis=0)          # total darkness of the original word
    coverage = ref[0].astype(float).sum() / 255.0                   # total coverage of the same word redrawn
    return np.clip(bg - mass / max(coverage, 1.0), 0, 255), bg

def word_geometry(w, info, font_path, size):
    """Where the original text sits: (ink_left, ink_right, baseline_y) in image pixels."""
    x0, y0, x1, y1 = w['px']
    ref = render_text(w['text'], font_path, size)
    if info["inkbox"] and ref is not None:
        ix0, iy0, ix1, iy1 = info["inkbox"]
        r_alpha, _r_left, r_top = ref
        baseline = int(round(((iy0 - r_top) + (iy1 - (r_top + r_alpha.shape[0]))) / 2.0))
        return ix0, ix1, baseline
    return x0, x1, y1 - int(round(0.2 * (y1 - y0)))

def draw_word(out, w, font_path, size, colour, left, baseline, squeeze=1.0):
    """Draw w['new_text'] with its ink starting at x=`left` on `baseline`."""
    new = render_text(w['new_text'], font_path, size)
    if new is None:
        return "replaced"
    patch, _n_left, n_top = new
    top = baseline + n_top
    note = "replaced"
    if squeeze < 0.995:                              # squeeze horizontally rather than shrink the font
        width = max(2, int(round(patch.shape[1] * squeeze)))
        patch = np.asarray(Image.fromarray(patch).resize((width, patch.shape[0]), Image.BILINEAR))
        note = "replaced (condensed)"

    im = Image.fromarray(np.pad(patch, 3))
    if abs(w['angle']) > ROTATE_MIN_DEG:
        centre_y = top + patch.shape[0] / 2.0
        im = im.rotate(-w['angle'], resample=Image.BICUBIC, expand=True)
        top = int(round(centre_y - im.size[1] / 2.0)) + 3
    if BLUR_RADIUS > 0:
        im = im.filter(ImageFilter.GaussianBlur(BLUR_RADIUS))
    alpha = np.asarray(im, dtype=float) / 255.0
    left, top = int(round(left)) - 3, top - 3

    H, W = out.shape[:2]
    ax0, ay0 = max(0, -left), max(0, -top)
    ax1, ay1 = min(alpha.shape[1], W - left), min(alpha.shape[0], H - top)
    if ax1 <= ax0 or ay1 <= ay0:
        return "outside image"
    a = alpha[ay0:ay1, ax0:ax1, None]
    region = out[top + ay0:top + ay1, left + ax0:left + ax1].astype(float)
    out[top + ay0:top + ay1, left + ax0:left + ax1] = np.clip(region * (1 - a) + colour * a, 0, 255).astype(np.uint8)
    return note


def _free_right(gray, x_from, y0, y1, x_to):
    """x of the first vertical form line between x_from and x_to on rows y0..y1 (x_to when there is none)."""
    x_from, x_to = max(0, int(x_from)), min(gray.shape[1], int(x_to))
    band = gray[max(0, y0):max(0, y1), x_from:x_to]
    if band.size == 0 or band.shape[1] < 2 or int(band.max()) - int(band.min()) < 40:
        return x_to
    dark = band <= _otsu(band)
    cols = np.flatnonzero(dark.mean(axis=0) >= 0.9)          # a column that is dark over the whole line height
    return x_from + int(cols[0]) if len(cols) else x_to


# ---------------------------------------------------------------- page level
def render_page(image_path, offset_map, flags):
    """Returns (PIL image, actions, doc_font). actions = {word_id: "replaced" | "boxed: reason" | "unchanged"}."""
    source = Image.open(image_path)
    keep_grey = source.mode in ("1", "L")           # save a grey scan as grey, not as a larger RGB file
    img = source.convert("RGB")
    orig = np.array(img)
    out = orig.copy()
    gray = np.asarray(img.convert("L"))
    rng = np.random.default_rng(0)
    flat = orig.reshape(-1, 3)
    bright = flat[gray.ravel() >= np.percentile(gray, 60)]
    page_bg = bright[:: max(1, len(bright) // 5000)]

    phi = [w for w in offset_map if w['word_id'] in flags]
    doc_font, line_styles = choose_styles(gray, offset_map, set(flags))

    # next word to the right on the same line, to know how much room there is
    next_x = {}
    by_line = {}
    for w in offset_map:
        by_line.setdefault(w['line_idx'], []).append(w)
    for ws in by_line.values():
        ws = sorted(ws, key=lambda w: w['px'][0])
        for a, b in zip(ws, ws[1:]):
            next_x[a['word_id']] = b['px'][0]

    actions, boxes, draws = {}, [], []
    for w in phi:
        if w.get('new_text', w['text']) == w['text']:
            actions[w['word_id']] = "unchanged"
            continue
        reason = None
        if w['handwriting']:
            reason = "handwriting"
        elif w['confidence'] < MIN_OCR_CONF:
            reason = "low OCR confidence"
        elif abs(w['angle']) > MAX_SKEW_DEG:
            reason = "rotated text"
        info = analyze_word(gray, w)
        erase_info = info
        if w['handwriting']:                           # erase a little wider for loops and tails (measuring stays tight)
            pad = max(2, int(round(HANDWRITING_ERASE_GROW * (w['px'][3] - w['px'][1]))))
            erase_info = analyze_word(gray, w, pad=pad) or info
            if erase_info is not None:
                erase_info["handwriting"] = True
        if reason == "low OCR confidence" and KEEP_LOW_CONFIDENCE:
            actions[w['word_id']] = "kept: low OCR confidence"
            #boxes.append(w)
            continue
        font_path, size = style_for_word(w, info, doc_font, line_styles)
        ink_left, ink_right, baseline = word_geometry(w, info, font_path, size)
        colour, _bg = ink_colour(orig, info, render_text(w['text'], font_path, size))
        new = render_text(w['new_text'], font_path, size)
        draws.append({"w": w, "info": info, "erase_info": erase_info, "font": font_path, "size": size, "colour": colour,
                      "ink_left": ink_left, "ink_right": ink_right, "baseline": baseline,
                      "new_w": new[0].shape[1] if new is not None else 0})

    def make_draw(w):
        """Measurements for redrawing an UNCHANGED word somewhere else on its line (carried along by a longer fake)."""
        info = analyze_word(gray, w)
        if info is None or not info["inkbox"]:
            return None
        font_path, size = style_for_word(w, info, doc_font, line_styles)
        ink_left, ink_right, baseline = word_geometry(w, info, font_path, size)
        colour, _bg = ink_colour(orig, info, render_text(w['text'], font_path, size))
        r = render_text(w['text'], font_path, size)
        w["new_text"] = w['text']                              # drawn as it was, only moved
        return {"w": w, "info": info, "erase_info": info, "font": font_path, "size": size, "colour": colour,
                "ink_left": ink_left, "ink_right": ink_right, "baseline": baseline,
                "new_w": r[0].shape[1] if r is not None else 0, "carried": True}

    line_words = {li: sorted(ws, key=lambda w: w['px'][0]) for li, ws in by_line.items()}

    # Lay out runs of consecutive replaced words: the first word starts where the original
    # started and the ORIGINAL gaps between words are kept, so typewriter text lands exactly
    # on the old positions and proportional text keeps natural spacing. If a run would reach
    # the next untouched word it is squeezed horizontally; if it is shorter, the spare room
    # is shared between its gaps.
    runs = []
    for d in draws:
        prev = runs[-1][-1] if runs else None
        if prev and prev["w"]['line_idx'] == d["w"]['line_idx'] and prev["w"]['idx'] == d["w"]['idx'] - 1 \
                and d["ink_left"] >= prev["ink_right"]:
            runs[-1].append(d)
        else:
            runs.append([d])
    page_w = gray.shape[1]

    def _layout(run):
        """Place the run's words at the original gaps; returns the right end of the last word."""
        x = run[0]["ink_left"]
        for k, d in enumerate(run):
            if k:
                x += run[k - 1]["new_w"] + (d["ink_left"] - run[k - 1]["ink_right"])
            d["left"] = x
        return run[-1]["left"] + run[-1]["new_w"]

    def _room(run, start):
        """Right limit for a run: MAX_EXTRA_WIDTH wider at most, never past the next word or a form line."""
        orig_right = run[-1]["ink_right"]
        last = run[-1]["w"]
        height = last['px'][3] - last['px'][1]
        gap = max(2, int(0.25 * height))
        next_left = next_x.get(last['word_id'], page_w)
        wall = _free_right(gray, orig_right + 1, last['px'][1], last['px'][3], next_left)
        limit = min(orig_right + MAX_EXTRA_WIDTH * (orig_right - start) + 0.5 * height, wall - gap, next_left - gap)
        return max(limit, orig_right), orig_right

    extra = []
    for run in runs:
        start, end = run[0]["ink_left"], _layout(run)
        limit, orig_right = _room(run, start)
        if CARRY_WORDS and end > limit:
            # the next words on the line are unchanged: move them along with the fake, up to a form line
            last = run[-1]["w"]
            ws = line_words.get(last['line_idx'], [])
            pos = next((k for k, x in enumerate(ws) if x['word_id'] == last['word_id']), None)
            height = last['px'][3] - last['px'][1]
            wall = _free_right(gray, orig_right + 1, last['px'][1], last['px'][3], page_w)
            carried = []
            for x in (ws[pos + 1:] if pos is not None else []):
                if x['word_id'] in flags or x['px'][2] > wall:     # stop at the next replaced word or a form line
                    break
                d = make_draw(x)
                if d is None:
                    break
                carried.append(d)
            if carried:
                run.extend(carried)
                extra.extend(carried)
                end = _layout(run)
                limit, orig_right = _room(run, start)
        ratio = (limit - start) / max(end - start, 1)
        scale = 1.0
        if ratio < MIN_SQUEEZE:                                   # too long even when condensed: smaller font
            scale = max(MIN_FONT_SCALE, ratio / MIN_SQUEEZE)
            for d in run:
                d["size"] = max(5, int(round(d["size"] * scale)))
                r = render_text(d["w"]['new_text'], d["font"], d["size"])
                d["new_w"] = r[0].shape[1] if r is not None else 0
            end = _layout(run)
            ratio = (limit - start) / max(end - start, 1)
        squeeze = min(1.0, ratio)
        slack = max(0.0, orig_right - end) if len(run) > 1 else 0.0   # shorter multi-word run: share the room between gaps
        for k, d in enumerate(run):
            d["left"] = start + (d["left"] - start) * squeeze + slack * k / len(run)
            d["squeeze"] = squeeze
            d["smaller"] = scale < 0.995
    draws += extra                                         # carried words are erased and redrawn too
    
    # erase everything first, then draw, so one word's eraser never touches another's new text
    for w in boxes:
        x0, y0, x1, y1 = w['px']
        out[max(0, y0):y1, max(0, x0):x1] = BOX_COLOR
    for d in draws:
        erase_word(out, orig, d["erase_info"], d["w"]['px'], rng, page_bg)
    for d in draws:
        note = draw_word(out, d["w"], d["font"], d["size"], d["colour"], d["left"], d["baseline"], d["squeeze"])
        if d.get("carried"):
            note = note.replace("replaced", "moved (unchanged text)", 1)
        if d.get("smaller"):
            note += " (smaller font)"
        if d["font"] in HANDWRITING_FONTS:
            note += " (handwriting font: " + os.path.basename(d["font"]) + ")"
        actions[d["w"]['word_id']] = note

    result = Image.fromarray(out)
    return (result.convert("L") if keep_grey else result), actions, doc_font


deid_image, actions, doc_font = render_page(IMAGE_PATH, offset_map, flags)

OUTPUT_IMAGES_DIR = os.path.join(os.path.dirname(IMAGE_PATH), "output_images")
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
_stem = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
deid_image_path = os.path.join(OUTPUT_IMAGES_DIR, f"{_stem}_deid.png")
deid_image.save(deid_image_path)      # new pixels only: no metadata from the original is carried over

_summary = _Counter(a.split(":")[0].split(" (")[0] for a in actions.values())
print(f"Main font used for the replacements: {os.path.basename(doc_font)}")
print(f"Words: {dict(_summary)}")
print(f"Saved: {deid_image_path}")
if DEBUG:
    debug_render(actions, word_by_id)

## 13b. Side-by-side Check (before the batch)

Shows the test image `IMAGE_PATH` next to its de-identified version (original left, de-identified right) and saves it as `<name>_comparison.png` in `output_images`. Check that every PHI value is replaced and that the fake values sit in the right place, in a matching font, before running the batch.

In [ ]:
# Original | de-identified, for the single test image (sections 4-13), before running the batch.
import matplotlib.pyplot as plt

def side_by_side(original_path, deid_img, label_band=56, gap=24):
    """One image: original on the left, de-identified on the right, each with a label above it."""
    original = Image.open(original_path).convert("RGB")
    deid_img = deid_img.convert("RGB")
    width, height = max(original.width, deid_img.width), max(original.height, deid_img.height)
    canvas = Image.new("RGB", (width * 2 + gap, height + label_band), "white")
    canvas.paste(original, (0, label_band))
    canvas.paste(deid_img, (width + gap, label_band))
    draw = ImageDraw.Draw(canvas)
    try:
        label_font = ImageFont.truetype(FONTS[0], 32)
    except Exception:
        label_font = ImageFont.load_default()
    draw.text((12, 10), "Original", fill="black", font=label_font)
    draw.text((width + gap + 12, 10), "De-identified", fill="black", font=label_font)
    return canvas

comparison = side_by_side(IMAGE_PATH, deid_image)
comparison_path = os.path.join(OUTPUT_IMAGES_DIR, f"{_stem}_comparison.png")
comparison.save(comparison_path)
if SAVE_TEXT_FILES:
    for _p in save_text_files(change_stats, offset_map, vlm_stats, ctx_text, OUTPUT_IMAGES_DIR, _stem) + \
              save_deid_texts(full_text, ctx_text, change_stats, OUTPUT_IMAGES_DIR, _stem):
        print(f"Saved: {_p}")
if SAVE_EXCEL_REPORT:
    print(f"Saved: {save_excel_report(IMAGE_PATH, full_text, offset_map, ctx_text, kv_pairs, vlm_stats, change_stats, OUTPUT_IMAGES_DIR, _stem)}")
print(f"Saved: {comparison_path}")
print(f"Changed values: {change_stats['changed_values']} | words: {dict(_summary)}")

plt.figure(figsize=(20, 12))
plt.imshow(comparison)
plt.axis("off")
plt.show()

## 14. Batch

`process_image_pipeline()` runs sections 4–13 for one page: OCR → word index → GPT-5.4 → Azure Surrogate (2 texts) + GPT verdict → redraw → save. A failed page is logged in `batch_errors.csv`; the loop continues. The "same value → same fake" memory is reset at the start of each batch.

**Files per page:** `<name>_deid.png` (`output_images` next to the input), `<name>_comparison.png` (`BATCH_COMPARISON_DIR`), `<name>_changes.txt` / `<name>_surrogated.txt` / `<name>_gpt.json` (next to `_deid.png`), `<name>_verify.json` (`VERIFY_DIR`). `_changes.txt`, `_gpt.json` and `_verify.json` hold real PHI: keep them protected.

**Summary columns:** `changed_words`, `changed_values`, `by_page_text`, `by_label_value_text`, `kept_gpt_not_phi`, `kept_protected`, `gpt_phi_not_changed_by_azure`, `dates_fixed`, `emails_realistic`, `condensed_words`, `smaller_font_words`, `moved_words`, `handwritten_phi_words`, `handwriting_font_words`, `vlm_used`, `vlm_error`, `deid_calls`, `deid_input_bytes`, `font_used`.

In [ ]:
from pathlib import Path
import csv

# Batch configuration: process every supported image under the input folder and save
# both the de-identified image and a side-by-side comparison image.
BATCH_INPUT_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\data")
BATCH_COMPARISON_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\compared_data")
BATCH_RECURSIVE = True

# Verification record per page, used by the phi_leak_verification notebook to find missed PHI.
# It holds the ORIGINAL PHI text next to its fake value, i.e. it is a re-identification key:
# keep this folder as protected as the original images and never share it with the outputs.
SAVE_VERIFICATION_RECORD = True
VERIFY_DIR = Path(r"C:\Users\RSahu\Documents\Azure_deidentification\Input_docs\verification_records")

def discover_images(root_dir, recursive=True):
    root_dir = Path(root_dir)
    pattern = "**/*" if recursive else "*"
    images = []
    for path in root_dir.glob(pattern):
        if not path.is_file():
            continue
        if path.parent.name.lower() == "output_images":
            continue
        if path.stem.lower().endswith("_deid") or path.stem.lower().endswith("_comparison"):
            continue
        if path.suffix.lower() in SUPPORTED_IMAGE_FORMATS:
            images.append(path)
    return sorted(images)

def save_comparison_image(original_path, deid_path, comparison_path):
    original = Image.open(original_path).convert("RGB")
    deidentified = Image.open(deid_path).convert("RGB")

    width = max(original.width, deidentified.width)
    height = max(original.height, deidentified.height)
    label_band = 48
    gap = 24

    canvas = Image.new("RGB", (width * 2 + gap, height + label_band), "white")
    canvas.paste(original, (0, label_band))
    canvas.paste(deidentified, (width + gap, label_band))

    draw = ImageDraw.Draw(canvas)
    label_font = ImageFont.load_default()
    draw.text((12, 14), "Original", fill="black", font=label_font)
    draw.text((width + gap + 12, 14), "De-identified", fill="black", font=label_font)

    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(comparison_path)
    return comparison_path

def _ocr_bytes_for(image):
    """Bytes for Textract (< 10 MB). A PNG re-save of a JPG scan can be larger than the
    original, so fall back to a high-quality JPEG for the OCR call only."""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    if buf.tell() < 10 * 1024 * 1024:
        return buf.getvalue()
    buf = io.BytesIO()
    image.convert("RGB").save(buf, format="JPEG", quality=95)
    return buf.getvalue()

def save_verification_record(image_path, deid_image, deid_image_path, offset_map, flags, entities, actions, record_path):
    """Re-OCR the de-identified image and save, for every word, the original text, whether it
    was flagged, its fake text and what happened on the image, plus the words read back from
    the output. Nothing is judged here; the verification notebook does that."""
    deid_blocks = get_ocr(_ocr_bytes_for(deid_image))["Blocks"]
    _deid_text, deid_words = build_word_index(deid_blocks, deid_image.width, deid_image.height)

    word_entity = {}
    for n, ent in enumerate(entities):
        for w in ent["words"]:
            word_entity[w['word_id']] = n

    record = {
        "image_path": str(image_path),
        "deid_image_path": str(deid_image_path),
        "image_size": [deid_image.width, deid_image.height],
        "original_words": [{
            "word_id": w['word_id'], "idx": w['idx'], "line_idx": w['line_idx'], "text": w['text'],
            "px": list(w['px']), "confidence": round(w['confidence'], 2), "handwriting": w['handwriting'],
            "flagged": w['word_id'] in flags,
            "kind": flags[w['word_id']]["kind"] if w['word_id'] in flags else None,
            "sources": sorted(flags[w['word_id']]["sources"]) if w['word_id'] in flags else [],
            "new_text": w.get('new_text') if w['word_id'] in flags else None,
            "action": actions.get(w['word_id']) if w['word_id'] in flags else None,
            "entity": word_entity.get(w['word_id']),
        } for w in offset_map],
        "entities": [{
            "n": n, "kind": ent["kind"], "original": ent["original"], "surrogate": ent.get("surrogate"),
            "sources": ent["sources"], "word_ids": [w['word_id'] for w in ent["words"]],
        } for n, ent in enumerate(entities)],
        "deid_words": [{
            "idx": w['idx'], "line_idx": w['line_idx'], "text": w['text'], "px": list(w['px']),
            "confidence": round(w['confidence'], 2),
        } for w in deid_words],
    }
    record_path.parent.mkdir(parents=True, exist_ok=True)
    record_path.write_text(json.dumps(record, indent=1), encoding="utf-8")
    return record_path

def process_image_pipeline(image_path, batch_input_dir, comparison_root):
    AZURE_USAGE.update(calls=0, input_bytes=0)          # count this page only
    image_path = Path(load_image(str(image_path)))
    with open(image_path, "rb") as file_handle:
        image_bytes = file_handle.read()
    assert len(image_bytes) < 10 * 1024 * 1024, f"Image exceeds 10MB sync limit: {image_path.name}"

    blocks = get_ocr(image_bytes)["Blocks"]                                # Textract
    with Image.open(image_path) as source_image:
        img_w, img_h = source_image.size
    full_text, offset_map = build_word_index(blocks, img_w, img_h)
    word_by_id = {entry["word_id"]: entry for entry in offset_map}

    kv_pairs, label_word_ids, ctx_text, ctx_map = [], set(), "", []        # GPT-5.4 context
    vlm_hits, vlm_label_ids, vlm_stats = [], set(), {}
    if USE_VLM:
        try:
            kv_pairs, label_word_ids, ctx_text, ctx_map, vlm_hits, vlm_label_ids, vlm_stats = vlm_build_context(
                image_path, offset_map, word_by_id)
        except Exception as exc:
            print(f"    ⚠ GPT step failed for {image_path.name}, continuing with the page text only: {exc!r}")
            vlm_stats = {"vlm_error": repr(exc)[:200]}
    protected_ids = protected_form_ids(offset_map, vlm_label_ids, gpt_value_ids(kv_pairs, vlm_hits))

    flags, entities, change_stats = decide_changes(full_text, offset_map, ctx_text, ctx_map,
                                                   kv_pairs, vlm_hits, protected_ids)          # Azure Surrogate
    deid_calls, deid_bytes = AZURE_USAGE["calls"], AZURE_USAGE["input_bytes"]
    if DEBUG_BATCH:
        print(f"\n##### {image_path.name}")
        debug_ocr(offset_map)
        debug_texts(full_text, ctx_text, change_stats["azure"], offset_map)
        debug_decisions(change_stats)

    deid_image, actions, doc_font = render_page(str(image_path), offset_map, flags)
    if DEBUG_BATCH:
        debug_render(actions, word_by_id)

    output_images_dir = image_path.parent / "output_images"
    output_images_dir.mkdir(parents=True, exist_ok=True)
    deid_image_path = output_images_dir / f"{image_path.stem}_deid.png"
    deid_image.save(deid_image_path)
    relative_dir = image_path.parent.relative_to(batch_input_dir) if image_path.parent != batch_input_dir else Path()
    comparison_path = comparison_root / relative_dir / f"{image_path.stem}_comparison.png"
    save_comparison_image(image_path, deid_image_path, comparison_path)
    if SAVE_TEXT_FILES:
        save_text_files(change_stats, offset_map, vlm_stats, ctx_text, output_images_dir, image_path.stem)
        save_deid_texts(full_text, ctx_text, change_stats, output_images_dir, image_path.stem)
    report_path = ""
    if SAVE_EXCEL_REPORT:
        report_path = save_excel_report(image_path, full_text, offset_map, ctx_text, kv_pairs, vlm_stats,
                                        change_stats, output_images_dir, image_path.stem)

    action_counts = Counter(action.split(":")[0].split(" (")[0] for action in actions.values())
    verify_path = ""
    if SAVE_VERIFICATION_RECORD:
        verify_path = VERIFY_DIR / relative_dir / f"{image_path.stem}_verify.json"
        save_verification_record(image_path, deid_image, deid_image_path, offset_map, flags,
                                 entities, actions, verify_path)

    row = {"image_path": str(image_path), "deid_image_path": str(deid_image_path), "comparison_path": str(comparison_path),
           "ocr_words": len(offset_map)}
    row.update({k: v for k, v in change_stats.items()
                if not k.endswith(("_rows", "_ids")) and k not in ("rows", "azure", "decision_kept", "_sources")})
    row["excel_report"] = str(report_path)
    row.update({
        "replaced_words": action_counts.get("replaced", 0),
        "condensed_words": sum(1 for a in actions.values() if "condensed" in a),
        "smaller_font_words": sum(1 for a in actions.values() if "smaller font" in a),
        "moved_words": sum(1 for a in actions.values() if a.startswith("moved")),
        "handwritten_phi_words": sum(1 for w in offset_map if w['word_id'] in flags and w['handwriting']),
        "handwriting_font_words": sum(1 for a in actions.values() if "handwriting font" in a),
        "vlm_used": bool(USE_VLM and "vlm_error" not in vlm_stats),
        "vlm_fields": vlm_stats.get("vlm_fields", 0), "vlm_fields_matched": vlm_stats.get("vlm_fields_matched", 0),
        "vlm_error": vlm_stats.get("vlm_error", ""),
        "font_used": os.path.basename(doc_font), "verification_record": str(verify_path),
        "deid_calls": deid_calls, "deid_input_bytes": deid_bytes,
    })
    return row

print(f"Batch input folder: {BATCH_INPUT_DIR}")
print(f"Comparison output folder: {BATCH_COMPARISON_DIR}")
print(f"Images discovered right now: {len(discover_images(BATCH_INPUT_DIR, recursive=BATCH_RECURSIVE))}")

In [ ]:
batch_images = discover_images(BATCH_INPUT_DIR, recursive=BATCH_RECURSIVE)
print(f"Found {len(batch_images)} image(s) under {BATCH_INPUT_DIR}")

batch_results = []
batch_errors = []
SURROGATE_MEMO.clear()        # fresh "same value -> same Azure fake" memory for this batch
BATCH_WORD_ROWS.clear()       # word decisions of all pages, for batch_changes.xlsx

for index, image_path in enumerate(batch_images, start=1):
    print(f"[{index}/{len(batch_images)}] Processing {image_path.name}")
    try:
        result = process_image_pipeline(image_path, BATCH_INPUT_DIR, BATCH_COMPARISON_DIR)
        batch_results.append(result)
        print(
            f"    saved deid={result['deid_image_path']} | comparison={result['comparison_path']} | "
            f"changed_values={result['changed_values']} | replaced_words={result['replaced_words']}"
        )
    except Exception as exc:
        batch_errors.append({"image_path": str(image_path), "error": repr(exc)})
        print(f"    FAILED: {exc!r}")

BATCH_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
summary_csv_path = BATCH_COMPARISON_DIR / "batch_summary.csv"
error_csv_path = BATCH_COMPARISON_DIR / "batch_errors.csv"

if batch_results:
    with open(summary_csv_path, "w", newline="", encoding="utf-8") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=list(batch_results[0].keys()))
        writer.writeheader()
        writer.writerows(batch_results)

if batch_errors:
    with open(error_csv_path, "w", newline="", encoding="utf-8") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=["image_path", "error"])
        writer.writeheader()
        writer.writerows(batch_errors)

print()
print(f"Completed: {len(batch_results)} image(s)")
print(f"Failed: {len(batch_errors)} image(s)")
print(f"Summary CSV: {summary_csv_path}")
if SAVE_EXCEL_REPORT:
    print(f"Batch Excel: {save_batch_excel(BATCH_COMPARISON_DIR / 'batch_changes.xlsx')}")
total_bytes = sum(r["deid_input_bytes"] for r in batch_results)
print(f"Azure DeID: {sum(r['deid_calls'] for r in batch_results)} calls, "
      f"{total_bytes:,} bytes = {total_bytes / 1_000_000:.3f} MB "
      f"(avg {total_bytes / max(len(batch_results), 1) / 1000:.1f} KB per page)")
if batch_errors:
    print(f"Error CSV:   {error_csv_path}")

batch_results[:3]

In [ ]:
# Preview one saved comparison image after the batch run.
import matplotlib.pyplot as plt
PREVIEW_INDEX = 0

if not batch_results:
    raise RuntimeError("No batch results available. Run the batch cell above first.")

preview_result = batch_results[PREVIEW_INDEX]
preview_path = preview_result["comparison_path"]
print(f"Previewing {preview_path}")

preview_img = Image.open(preview_path).convert("RGB")
plt.figure(figsize=(18, 10))
plt.imshow(preview_img)
plt.axis("off")
plt.show()